# Task 11 — Recommendation Systems Foundations & Content-Based Filtering

**Goal:** frame a recommendation problem correctly (queries, items, the feedback matrix, explicit vs. implicit signals), then build a first working recommender with **content-based filtering**: represent every movie as a feature vector and recommend by similarity.

**Dataset:** MovieLens `ml-latest-small` (GroupLens): `movies.csv`, `ratings.csv`, `tags.csv`.

| # | Section |
|---|---|
| 0 | Setup & data loading |
| 1 | What is a recommendation system? |
| 2 | The feedback matrix: explicit vs. implicit signals |
| 3 | Content-based filtering: core idea & item profiles |
| 4 | Representing text items: TF-IDF |
| 5 | Cosine similarity & ranking items |
| 6 | Building user profiles from item features |
| 7 | Strengths & weaknesses of content-based filtering |
| 8 | Capstone: end-to-end content-based movie recommender |
| E | Exercises 1–5 |
| ✔ | Deliverables checklist |

> **Why `ml-latest-small`?** The readme allows "MovieLens 100K or 25M". The classic `ml-100k` ships `u.data` / `u.item`, not the `movies.csv` / `ratings.csv` format the readme's code uses. `ml-latest-small` has exactly that format, has ~100K ratings, and also includes `tags.csv`. The 25M version is ~250 MB, which is too heavy to carry through all five tasks of this track. The same dataset will be reused in Tasks 12–15.

## 0. Setup & Data Loading

Environment check from the readme (`python3 -c "import numpy, pandas, sklearn"`), plus the imports used throughout.

In [1]:
import time
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 70)

print("numpy  :", np.__version__)
print("pandas :", pd.__version__)
print("sklearn:", sklearn.__version__)

numpy  : 2.5.3
pandas : 3.0.5
sklearn: 1.9.1


In [2]:
# Find the extracted MovieLens folder next to this notebook
candidates = [Path("ml-latest-small"), Path("ml-latest-small/ml-latest-small"), Path(".")]
DATA_DIR = next((p for p in candidates if (p / "movies.csv").exists()), None)
assert DATA_DIR is not None, "movies.csv not found - extract ml-latest-small.zip next to this notebook"

movies  = pd.read_csv(DATA_DIR / "movies.csv")    # movieId, title, genres
ratings = pd.read_csv(DATA_DIR / "ratings.csv")   # userId, movieId, rating, timestamp
tags    = pd.read_csv(DATA_DIR / "tags.csv")      # userId, movieId, tag, timestamp

print(f"Data folder: {DATA_DIR}")
print(f"movies : {movies.shape}")
print(f"ratings: {ratings.shape}")
print(f"tags   : {tags.shape}")
print(f"\nusers: {ratings.userId.nunique()} | movies with >=1 rating: {ratings.movieId.nunique()} "
      f"| movies in catalog: {movies.movieId.nunique()}")
display(movies.head())
display(ratings.head())

Data folder: ml-latest-small
movies : (9742, 3)
ratings: (100836, 4)
tags   : (3683, 4)

users: 610 | movies with >=1 rating: 9724 | movies in catalog: 9742


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
# Data-quality checks that later code relies on
print("Duplicate (userId, movieId) pairs in ratings:", ratings.duplicated(["userId", "movieId"]).sum())
print("Duplicate movieIds in movies.csv           :", movies.movieId.duplicated().sum())
print("Rated movieIds missing from movies.csv     :", (~ratings.movieId.isin(movies.movieId)).sum())
print("Missing values (movies / ratings / tags)   :",
      movies.isna().sum().sum(), "/", ratings.isna().sum().sum(), "/", tags.isna().sum().sum())

dup_titles = movies[movies.title.duplicated(keep=False)].sort_values("title")
print(f"\nTitles that appear more than once: {dup_titles.title.nunique()}")
display(dup_titles)

Duplicate (userId, movieId) pairs in ratings: 0
Duplicate movieIds in movies.csv           : 0
Rated movieIds missing from movies.csv     : 0
Missing values (movies / ratings / tags)   : 0 / 0 / 0

Titles that appear more than once: 5


,movieId,title,genres
4169,6003,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Thriller
9106,144606,Confessions of a Dangerous Mind (2002),Comedy|Crime|Drama|Romance|Thriller
650,838,Emma (1996),Comedy|Drama|Romance
5601,26958,Emma (1996),Romance
5854,32600,Eros (2004),Drama
9135,147002,Eros (2004),Drama|Romance
2141,2851,Saturn 3 (1980),Adventure|Sci-Fi|Thriller
9468,168358,Saturn 3 (1980),Sci-Fi|Thriller
5931,34048,War of the Worlds (2005),Action|Adventure|Sci-Fi|Thriller
6932,64997,War of the Worlds (2005),Action|Sci-Fi


In [4]:
# Lookup tables: row position in `movies` <-> movieId / title.
# Every item matrix built below has one row per movie in this exact order.
movies = movies.reset_index(drop=True)
movieid_to_idx = pd.Series(movies.index, index=movies["movieId"])
title_to_idx = pd.Series(movies.index, index=movies["title"])
title_to_idx = title_to_idx[~title_to_idx.index.duplicated(keep="first")]   # duplicate titles -> first one

print("movieid_to_idx entries:", len(movieid_to_idx))
print("title_to_idx entries  :", len(title_to_idx))

movieid_to_idx entries: 9742
title_to_idx entries  : 9737


**Notes:** every `(userId, movieId)` pair is unique, so `nnz = len(ratings)` is valid in Phase 2. Every rated movie exists in `movies.csv`. A few titles appear twice with different `movieId`s; title lookups below use the first one, and anything that matters is keyed by `movieId`.

---
# Phase 1 — What Is a Recommendation System?

### 1.1 Definition
A **recommendation system** predicts, for a given query, which items a user is most likely to engage with, out of a (often huge) catalog.

### 1.2 Query (also called *context*)
The **query** is the input the recommender works from. It can be just a user ID, or a richer combination: the user's features and history, the item they are viewing right now, the time of day, device, and location. Example: *"user 42, on a phone, at 11 pm, currently watching a thriller"*.

### 1.3 Item
The **item** is the entity being recommended: movies, products, videos, jobs, songs, news articles. In this task the items are the movies in `movies.csv`.

### 1.4 Real-world recommenders: query / item / algorithm family

In [5]:
real_world = pd.DataFrame([
    ["Netflix", "Streaming", "User + watch history + device/time", "Movie / show",
     "Hybrid: viewing behaviour of similar users + title metadata (genre, cast)"],
    ["YouTube", "Video", "User + watch/search history + video being watched", "Video",
     "Hybrid: co-watch behaviour + video features, ranked by deep models"],
    ["Amazon", "E-commerce", "User + purchase/browse history + product being viewed", "Product",
     "Mainly collaborative ('customers who bought X also bought Y') + product attributes"],
    ["Spotify", "Music", "User + listening history + time of day", "Song / playlist",
     "Hybrid: co-listening patterns + audio and text features of tracks"],
    ["LinkedIn Jobs", "Recruiting", "Member profile (skills, title, location)", "Job posting",
     "Hybrid: profile-to-job content matching + what similar members applied to"],
], columns=["Platform", "Industry", "Query (context)", "Item", "Likely algorithm family"])
real_world

,Platform,Industry,Query (context),Item,Likely algorithm family
0,Netflix,Streaming,User + watch history + device/time,Movie / show,Hybrid: viewing behaviour of similar users + title metadata (genre...
1,YouTube,Video,User + watch/search history + video being watched,Video,"Hybrid: co-watch behaviour + video features, ranked by deep models"
2,Amazon,E-commerce,User + purchase/browse history + product being viewed,Product,Mainly collaborative ('customers who bought X also bought Y') + pr...
3,Spotify,Music,User + listening history + time of day,Song / playlist,Hybrid: co-listening patterns + audio and text features of tracks
4,LinkedIn Jobs,Recruiting,"Member profile (skills, title, location)",Job posting,Hybrid: profile-to-job content matching + what similar members app...


### 1.5 The three algorithm families and the data each needs

| Family | Data required | Core idea | Example |
|---|---|---|---|
| **Content-based filtering** | Item **features** + the target user's own history. No other users needed. | Recommend items whose features resemble items this user already liked | "You liked *Toy Story* (Animation, Children) → here is *Monsters, Inc.*" |
| **Collaborative filtering** | The **user-item interaction matrix** only. No item features needed. | Users who agreed in the past will agree in the future | "Users who liked what you liked also liked *Shrek*" |
| **Hybrid** | **Both** item features and the interaction matrix | Combine the two, so each covers the other's blind spots | Netflix, YouTube, Spotify |

### 1.6 Recommendation vs. search: the zero-query problem

Search starts from an explicit query: the user types *"space movie with Tom Hanks"*, so the system knows exactly what is wanted and only has to retrieve and rank matches. Recommendation usually has **no explicit query at all**. When someone opens Netflix, the home page has to be filled before they type anything. The system must infer intent purely from indirect evidence (past ratings, clicks, watches, time, device) and guess what the user will want out of thousands of items. This is the **zero-query problem**: the "query" is reconstructed from context and history instead of being stated. That is why framing the query correctly (what context do we have, and what counts as a positive signal) is the first design decision of any recommender, before any modelling.

**✔ Phase 1 deliverable:** the 5-platform query/item/algorithm table above, plus the zero-query explanation.

---
# Phase 2 — The Feedback Matrix: Explicit vs. Implicit Signals

The **feedback matrix** `A ∈ ℝ^(m×n)` has one row per user (query) and one column per item. Cell `A[u, i]` holds user `u`'s feedback on item `i`, or is empty if they never interacted. Every recommender in Tasks 11–15 is built on this structure.

### 2.1 Build the feedback matrix from `ratings.csv`

In [6]:
feedback_matrix = ratings.pivot(index="userId", columns="movieId", values="rating")
m, n = feedback_matrix.shape
nnz = ratings.shape[0]                 # valid: every (userId, movieId) pair is unique (checked in Setup)
sparsity = 1 - nnz / (m * n)

print(f"Feedback matrix A: {m} users x {n:,} movies = {m * n:,} cells")
print(f"Observed ratings (nnz): {nnz:,}")
print(f"Sparsity = 1 - nnz/(m*n) = {sparsity:.4%}")
print(f"Cross-check, non-empty cells counted in the matrix itself: {int(feedback_matrix.notna().sum().sum()):,}")

# Against the full catalog (including movies nobody rated) it is slightly sparser still
n_catalog = movies.movieId.nunique()
print(f"Sparsity vs full catalog ({n_catalog:,} movies): {1 - nnz / (m * n_catalog):.4%}")

feedback_matrix.iloc[:8, :12]

Feedback matrix A: 610 users x 9,724 movies = 5,931,640 cells
Observed ratings (nnz): 100,836
Sparsity = 1 - nnz/(m*n) = 98.3000%
Cross-check, non-empty cells counted in the matrix itself: 100,836
Sparsity vs full catalog (9,742 movies): 98.3032%


movieId,1,2,3,4,5,6,7,8,9,10,11,12
userId,,,,,,,,,,,,
1,4.0,NaN,4.0,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,4.0,5.0,3.0,5.0,4.0,4.0,3.0,NaN,3.0,4.0,NaN
7,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,4.0,NaN


In [7]:
per_user = ratings.groupby("userId").size()
per_movie = ratings.groupby("movieId").size()
print("Ratings per user : min", per_user.min(), "| median", int(per_user.median()), "| max", per_user.max())
print("Ratings per movie: min", per_movie.min(), "| median", int(per_movie.median()), "| max", per_movie.max())
print(f"Movies with only 1 rating: {(per_movie == 1).sum():,} ({(per_movie == 1).mean():.1%} of rated movies)")
print(f"Average user has rated {per_user.mean() / n:.2%} of the movies")

Ratings per user : min 20 | median 70 | max 2698
Ratings per movie: min 1 | median 3 | max 329
Movies with only 1 rating: 3,446 (35.4% of rated movies)
Average user has rated 1.70% of the movies


**Reading the matrix:** about 98.3% of the cells are empty. The average user has touched under 2% of the catalog, and many movies have a single rating. A recommender's whole job is to fill in this mostly-empty matrix.

### 2.2 Explicit vs. implicit feedback

| | **Explicit feedback** | **Implicit feedback** |
|---|---|---|
| What it is | The user directly states a preference | Preference is *inferred* from behaviour |
| Examples | Star rating, thumbs up/down, written review | Click, watch, purchase, time spent, skip, add-to-cart |
| Strength | Direct, unambiguous signal (5★ means liked) | Abundant: produced by every interaction automatically |
| Weakness | **Sparse** (few people bother to rate) and has **selection bias** (users only rate items they chose to engage with) | **Noisy**: a click doesn't mean "liked" (clickbait, accidental play), and "no click" doesn't mean "disliked" (maybe they never saw it) |

The selection bias is visible in this data. People mostly watch, and therefore rate, movies they expected to like, so the rating distribution is skewed high:

In [8]:
dist = ratings.rating.value_counts().sort_index()
display(pd.DataFrame({"count": dist, "share": (dist / dist.sum()).round(3)}).T)
print(f"Mean rating: {ratings.rating.mean():.2f} on a 0.5-5 scale")
print(f"Share of ratings >= 3.0: {(ratings.rating >= 3).mean():.1%}  |  share >= 4.0: {(ratings.rating >= 4).mean():.1%}")

rating,0.5,1.0,1.5,2.0,2.5,3.0,3.5,4.0,4.5,5.0
count,1370.000,2811.000,1791.000,7551.000,5550.000,20047.000,13136.00,26818.000,8551.000,13211.000
share,0.014,0.028,0.018,0.075,0.055,0.199,0.13,0.266,0.085,0.131


Mean rating: 3.50 on a 0.5-5 scale
Share of ratings >= 3.0: 81.1%  |  share >= 4.0: 48.2%


### 2.3 Convert explicit ratings into an implicit binary matrix

Rule from the readme: `1` if `rating ≥ 4` ("liked"), otherwise **unobserved** (left empty, *not* 0).

In [9]:
LIKE_THRESHOLD = 4.0

implicit_matrix = (feedback_matrix >= LIKE_THRESHOLD).astype(float).where(feedback_matrix >= LIKE_THRESHOLD)
nnz_implicit = int((ratings.rating >= LIKE_THRESHOLD).sum())
sparsity_implicit = 1 - nnz_implicit / (m * n)

print(f"Explicit matrix: {nnz:,} observed cells -> sparsity {sparsity:.4%}")
print(f"Implicit matrix: {nnz_implicit:,} observed cells -> sparsity {sparsity_implicit:.4%}")
print(f"Ratings below {LIKE_THRESHOLD} that became 'unobserved': {nnz - nnz_implicit:,} "
      f"({(nnz - nnz_implicit) / nnz:.1%} of all ratings)")
print(f"Users with zero positives in the implicit view : {int((implicit_matrix.notna().sum(axis=1) == 0).sum())}")
print(f"Rated movies with zero positives (empty column): {int((implicit_matrix.notna().sum(axis=0) == 0).sum()):,}")

Explicit matrix: 100,836 observed cells -> sparsity 98.3000%
Implicit matrix: 48,580 observed cells -> sparsity 99.1810%
Ratings below 4.0 that became 'unobserved': 52,256 (51.8% of all ratings)


Users with zero positives in the implicit view : 1
Rated movies with zero positives (empty column): 3,426


In [10]:
# Same user, same movies, two framings
u = per_user.index[0]
sample_cols = ratings[ratings.userId == u].sort_values("rating").movieId.values[[0, 1, 2, -3, -2, -1]]
side_by_side = pd.DataFrame({
    "title": movies.set_index("movieId").loc[sample_cols, "title"].values,
    "explicit rating": feedback_matrix.loc[u, sample_cols].values,
    "implicit (>=4)": implicit_matrix.loc[u, sample_cols].values,
}, index=sample_cols)
print(f"User {u}:")
side_by_side

User 1:


,title,explicit rating,implicit (>=4)
3176,"Talented Mr. Ripley, The (1999)",1.0,NaN
1219,Psycho (1960),2.0,NaN
2253,Toys (1992),2.0,NaN
3729,Shaft (1971),5.0,1.0
3793,X-Men (2000),5.0,1.0
5060,M*A*S*H (a.k.a. MASH) (1970),5.0,1.0


**How the two framings treat the same raw data:** in the explicit matrix, a 2★ rating is a *known dislike*, which is useful information. In the implicit view the same cell becomes "unobserved", indistinguishable from a movie the user has never heard of. The implicit framing throws away negative information and about 52% of the observed cells. That is also why it is sparser (99.18% vs 98.30%). In return it gives a clean "liked / unknown" signal, which matches what real logs (clicks, watches) look like. One user has no rating ≥ 4 at all, so in the implicit view they have no positives; they come back in Exercise 4.

### 2.4 Why production systems run mostly on implicit feedback

YouTube, Spotify and Amazon get maybe one explicit rating for every thousands of plays, clicks and purchases. Implicit signals are produced by every interaction for free, so they are **orders of magnitude more abundant** and cover far more users and items. They are also available for users who never rate anything. A huge, noisy signal beats a tiny, clean one, and the noise can be partly handled with techniques like confidence weighting (watch time, repeat plays) in later tasks.

**✔ Phase 2 deliverable:** pivoted feedback matrix, sparsity percentage, the implicit binary version, and the explicit-vs-implicit write-up.

---
# Phase 3 — Content-Based Filtering: Core Idea & Item Profiles

### 3.1 Core idea
**Content-based filtering recommends items whose *features* resemble the features of items this specific user already liked. No other users' data is involved.**

### 3.2 Item profiles: multi-hot genre vectors
Each movie's pipe-separated `genres` string becomes a multi-hot vector: one column per genre, 1 if present.

In [11]:
genres_dummies = movies["genres"].str.get_dummies(sep="|")
item_profiles = pd.concat([movies[["movieId", "title"]], genres_dummies], axis=1)

print("Item-profile table shape:", item_profiles.shape)
print("Genre columns:", list(genres_dummies.columns))
item_profiles.head()

Item-profile table shape: (9742, 22)
Genre columns: ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,movieId,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,Toy Story (1995),0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [12]:
print("Movies per genre:")
display(genres_dummies.sum().sort_values(ascending=False).to_frame("movies").T)
print("Genres per movie:", genres_dummies.sum(axis=1).value_counts().sort_index().to_dict())

Movies per genre:


,Drama,Comedy,Thriller,Action,Romance,Adventure,Crime,Sci-Fi,Horror,Fantasy,Children,Animation,Mystery,Documentary,War,Musical,Western,IMAX,Film-Noir,(no genres listed)
movies,4361,3756,1894,1828,1596,1263,1199,980,978,779,664,611,573,440,382,334,167,158,87,34


Genres per movie: {1: 2851, 2: 3218, 3: 2338, 4: 987, 5: 271, 6: 63, 7: 12, 8: 1, 10: 1}


`(no genres listed)` is a placeholder for missing data, not a real genre. If it were kept as a feature, every genre-less movie would look identical to every other one. It is dropped from the modelling vectors, so those movies get an all-zero vector, which honestly means "no content information".

In [13]:
GENRE_COLS = [g for g in genres_dummies.columns if g != "(no genres listed)"]
print(f"{len(GENRE_COLS)} real genres used as features")

19 real genres used as features


### 3.3 Adding a numeric feature: release year (parsed from the title)

In [14]:
movies["year"] = movies["title"].str.extract(r"\((\d{4})\)\s*$", expand=False).astype(float)
n_missing_year = int(movies.year.isna().sum())
print(f"Movies without a parsable year: {n_missing_year}")
display(movies.loc[movies.year.isna(), ["title", "genres"]].head(10))

YEAR_FILL = movies["year"].median()
movies["year_filled"] = movies["year"].fillna(YEAR_FILL)
print(f"Year range: {int(movies.year.min())}-{int(movies.year.max())}, missing filled with median {int(YEAR_FILL)}")

Movies without a parsable year: 13


,title,genres
6059,Babylon 5,Sci-Fi
9031,Ready Player One,Action|Sci-Fi|Thriller
9091,Hyena Road,(no genres listed)
9138,The Adventures of Sherlock Holmes and Doctor Watson,(no genres listed)
9179,Nocturnal Animals,Drama|Thriller
9259,Paterson,(no genres listed)
9367,Moonlight,Drama
9448,The OA,(no genres listed)
9514,Cosmos,(no genres listed)
9515,Maria Bamford: Old Baby,(no genres listed)


Year range: 1902-2018, missing filled with median 1999


**Should the year be scaled before combining it with the 0/1 genre columns? Yes, and here's why.** Genre values are 0 or 1, while a raw year is about 2000. In a cosine-similarity calculation the year would then make up >99.99% of every vector's length, so *every* movie would point in almost exactly the same direction and look nearly identical. Demonstration with two movies that share no genre:

In [15]:
a, b = title_to_idx["Toy Story (1995)"], title_to_idx["Shining, The (1980)"]
G01 = genres_dummies[GENRE_COLS].to_numpy(dtype=float)

raw = np.column_stack([G01, movies["year_filled"].to_numpy()])
year_scaled = MinMaxScaler().fit_transform(movies[["year_filled"]]).ravel()
scaled = np.column_stack([G01, year_scaled])

print(movies.loc[a, "title"], "|", movies.loc[a, "genres"])
print(movies.loc[b, "title"], "|", movies.loc[b, "genres"])
print(f"cosine, genres only           : {cosine_similarity(G01[[a]], G01[[b]])[0, 0]:.4f}")
print(f"cosine, genres + RAW year     : {cosine_similarity(raw[[a]], raw[[b]])[0, 0]:.4f}   <- year swamps everything")
print(f"cosine, genres + SCALED year  : {cosine_similarity(scaled[[a]], scaled[[b]])[0, 0]:.4f}")

Toy Story (1995) | Adventure|Animation|Children|Comedy|Fantasy
Shining, The (1980) | Horror
cosine, genres only           : 0.0000
cosine, genres + RAW year     : 1.0000   <- year swamps everything
cosine, genres + SCALED year  : 0.1883


Min-max scaling squeezes the year into [0, 1], the same range as a genre flag, so it no longer swamps the genres. It still adds some similarity on its own: two movies with *no genre in common* now score 0.19 just because both are from the 1980s–90s. So how much weight the year gets relative to genres matters; that is tested in **Exercise 1**. The full item-profile table, with the scaled year added:

In [16]:
item_profiles["year"] = movies["year"]
item_profiles["year_scaled"] = year_scaled.round(4)
print("Full item-profile table:", item_profiles.shape)
item_profiles.sample(5, random_state=0)

Full item-profile table: (9742, 24)


,movieId,title,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,year,year_scaled
272,313,"Swan Princess, The (1994)",0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1994.0,0.7931
5665,27618,"Sound of Thunder, A (2005)",0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,0,0,2005.0,0.8879
3030,4054,Save the Last Dance (2001),0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,2001.0,0.8534
4015,5673,Punch-Drunk Love (2002),0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,2002.0,0.8621
8557,116138,Leviathan (2014),0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,2014.0,0.9655


### 3.4 One movie's full feature vector

In [17]:
seed_title = "Toy Story (1995)"
row = item_profiles.loc[title_to_idx[seed_title]]
print(f"{seed_title} - genres string: {movies.loc[title_to_idx[seed_title], 'genres']}\n")
print(row.drop(["movieId", "title"]).to_string())

active = [g for g in genres_dummies.columns if row[g] == 1]
expected = movies.loc[title_to_idx[seed_title], "genres"].split("|")
assert sorted(active) == sorted(expected)
print("\nGenres with a 1 in the vector:", active, "-> matches the genres string ✔")

Toy Story (1995) - genres string: Adventure|Animation|Children|Comedy|Fantasy

(no genres listed)         0
Action                     0
Adventure                  1
Animation                  1
Children                   1
Comedy                     1
Crime                      0
Documentary                0
Drama                      0
Fantasy                    1
Film-Noir                  0
Horror                     0
IMAX                       0
Musical                    0
Mystery                    0
Romance                    0
Sci-Fi                     0
Thriller                   0
War                        0
Western                    0
year                  1995.0
year_scaled           0.8017

Genres with a 1 in the vector: ['Adventure', 'Animation', 'Children', 'Comedy', 'Fantasy'] -> matches the genres string ✔


**Explanation:** *Toy Story* has a 1 in exactly the five genre columns from its genre string (Adventure, Animation, Children, Comedy, Fantasy) and 0 in the other 15, plus its release year (1995) and the scaled version. This vector is the movie's entire identity as far as a content-based recommender is concerned. Two movies are "similar" only if these vectors are similar.

### 3.5 Why content-based filtering solves *item* cold start
A brand-new movie can be recommended on day one because its vector is built **only from its own features** (genres, year, description). It needs zero ratings or interactions from any other user before it can be compared to what a user likes.

**✔ Phase 3 deliverable:** full multi-hot item-profile table for all 9,742 movies, plus the single-movie feature vector with explanation.

---
# Phase 4 — Representing Text Items: TF-IDF

### 4.1 Why raw word counts are a poor representation
With plain counts, the most frequent words dominate each vector, and those are usually generic filler ("the", "movie", "story") that appears in *every* description and says nothing about what makes an item different. A small demo with four made-up plot descriptions:

In [18]:
docs = [
    "the movie is the story of a toy cowboy and a space ranger",
    "the movie is the story of a shark that attacks a beach town",
    "the movie is the story of a space crew hunted by an alien",
    "the movie is the story of a cowboy seeking revenge in the west",
]
cv = CountVectorizer()
C = cv.fit_transform(docs).toarray()
vocab_toy = cv.get_feature_names_out()

# (1) raw counts  (2) the readme formula TF x log(N/df), computed by hand  (3) sklearn TfidfVectorizer
df_toy = (C > 0).sum(axis=0)
readme_tfidf = C * np.log(len(docs) / df_toy)
sk = TfidfVectorizer(stop_words="english")
S = pd.DataFrame(sk.fit_transform(docs).toarray(), columns=sk.get_feature_names_out())

doc0 = pd.DataFrame({
    "raw count": C[0],
    "readme TF-IDF  TF*log(N/df)": readme_tfidf[0].round(3),
    "sklearn TF-IDF (stop words removed)": S.reindex(columns=vocab_toy).loc[0].round(3).values,
}, index=vocab_toy)
doc0 = doc0[doc0["raw count"] > 0].sort_values("raw count", ascending=False)
print("Doc 0:", docs[0])
display(doc0)

def cos01(M):
    return cosine_similarity(np.asarray(M)[[0]], np.asarray(M)[[1]])[0, 0]
print("Cosine similarity, doc 0 (toy film) vs doc 1 (shark film), which share only filler words:")
print(f"  raw counts           : {cos01(C):.3f}")
print(f"  readme TF-IDF        : {cos01(readme_tfidf):.3f}")
print(f"  sklearn TF-IDF       : {cos01(S):.3f}")

Doc 0: the movie is the story of a toy cowboy and a space ranger


,raw count,readme TF-IDF TF*log(N/df),sklearn TF-IDF (stop words removed)
the,2,0.000,NaN
and,1,1.386,NaN
cowboy,1,0.693,0.405
is,1,0.000,NaN
of,1,0.000,NaN
movie,1,0.000,0.268
ranger,1,1.386,0.514
space,1,0.693,0.405
story,1,0.000,0.268
toy,1,1.386,0.514


Cosine similarity, doc 0 (toy film) vs doc 1 (shark film), which share only filler words:
  raw counts           : 0.615
  readme TF-IDF        : 0.000
  sklearn TF-IDF       : 0.131


**What this shows:**
- **Raw counts:** "the" is the biggest feature, and filler words ("movie", "is", "story", "of") weigh as much as "toy" or "ranger". The toy film and the shark film look quite similar purely because of shared filler.
- **Readme TF-IDF:** every word that appears in *all* documents gets `log(N/df) = log(1) = 0`, so filler is wiped out. "toy" and "ranger" (in only one document) get the highest weight, and "cowboy"/"space" (in two documents) get half that. ("and" also scores high only because it happens to appear in just this one document, which is why stop-word lists are still useful.) The two unrelated films now have similarity 0.
- **sklearn TF-IDF:** sklearn's smoothed IDF (next section) never reaches exactly 0, which is why the readme also passes `stop_words="english"` to drop "the/is/of". The remaining filler ("movie", "story") still gets the lowest weight.

### 4.2 The TF-IDF formula

```
TF-IDF(t, d) = TF(t, d) × IDF(t)
   TF(t, d) = count of term t in document d
   IDF(t)   = log(N / df(t))
   N        = total number of documents (items)
   df(t)    = number of documents containing term t
```

**What scikit-learn actually computes** (worth knowing, so the numbers can be explained): by default `TfidfVectorizer` uses a *smoothed* IDF, `IDF(t) = ln((1 + N) / (1 + df(t))) + 1`. The `+1`s avoid division by zero and stop a term that appears in every document from getting weight 0. It then **L2-normalizes each row** so every item vector has length 1. Same idea as the textbook formula, slightly different numbers.

### 4.3 What each factor corrects for
- **TF (term frequency)** rewards terms that are central to *this* item: a word repeated in a description is probably what it's about.
- **IDF (inverse document frequency)** punishes terms that are so common across the catalog that they can't distinguish one item from another. A term in every document gets the lowest weight; a rare term gets a high one.

### 4.4 Fitting TF-IDF on the genres field

**First, exactly as written in the readme:**

In [19]:
movies["genres_text"] = movies["genres"].str.replace("|", " ", regex=False)
tfidf_naive = TfidfVectorizer(stop_words="english")
tfidf_naive_matrix = tfidf_naive.fit_transform(movies["genres_text"])
print("Shape:", tfidf_naive_matrix.shape)
print("Vocabulary:", list(tfidf_naive.get_feature_names_out()))

Shape: (9742, 23)
Vocabulary: ['action', 'adventure', 'animation', 'children', 'comedy', 'crime', 'documentary', 'drama', 'fantasy', 'fi', 'film', 'genres', 'horror', 'imax', 'listed', 'musical', 'mystery', 'noir', 'romance', 'sci', 'thriller', 'war', 'western']


The default tokenizer splits on punctuation, so the vocabulary has **artifacts**: `Sci-Fi` becomes `sci` + `fi`, `Film-Noir` becomes `film` + `noir`, and the placeholder `(no genres listed)` becomes the fake terms `genres` + `listed`. Each genre should be one token, so the fix is a tokenizer that splits on `|` and drops the placeholder:

In [20]:
def genre_tokenizer(genre_string):
    """'Adventure|Animation|Children' -> ['Adventure', 'Animation', 'Children']; drops the placeholder."""
    return [g for g in genre_string.split("|") if g != "(no genres listed)"]

tfidf = TfidfVectorizer(tokenizer=genre_tokenizer, token_pattern=None, lowercase=False, dtype=np.float32)
tfidf_matrix = tfidf.fit_transform(movies["genres"])
genre_vocab = tfidf.get_feature_names_out()

print("TF-IDF matrix shape:", tfidf_matrix.shape, "(n_movies, vocabulary_size)")
print("Vocabulary size    :", len(genre_vocab))
print("Vocabulary         :", list(genre_vocab))

TF-IDF matrix shape: (9742, 19) (n_movies, vocabulary_size)
Vocabulary size    : 19
Vocabulary         : ['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


In [21]:
# Verify sklearn's smoothed IDF by hand, and compare with the readme's log(N/df)
N = tfidf_matrix.shape[0]
df_t = np.asarray((tfidf_matrix > 0).sum(axis=0)).ravel()
idf_manual = np.log((1 + N) / (1 + df_t)) + 1
print("Manual smoothed IDF == sklearn idf_:", np.allclose(idf_manual, tfidf.idf_))

idf_table = pd.DataFrame({
    "df (movies with genre)": df_t,
    "IDF sklearn ln((1+N)/(1+df))+1": tfidf.idf_.round(3),
    "IDF readme log(N/df)": np.log(N / df_t).round(3),
}, index=genre_vocab).sort_values("IDF sklearn ln((1+N)/(1+df))+1", ascending=False)
idf_table

Manual smoothed IDF == sklearn idf_: True


,df (movies with genre),IDF sklearn ln((1+N)/(1+df))+1,IDF readme log(N/df)
Film-Noir,87,5.707,4.718
IMAX,158,5.115,4.122
Western,167,5.060,4.066
Musical,334,4.370,3.373
War,382,4.236,3.239
Documentary,440,4.095,3.097
Mystery,573,3.832,2.833
Animation,611,3.768,2.769
Children,664,3.685,2.686
Fantasy,779,3.525,2.526


The IDF ranking is what you'd expect: **Drama** (in ~4,400 movies) gets the lowest weight because it barely distinguishes anything, while rare genres like **Film-Noir** and **IMAX** get the highest. Both formulas give the same ordering; sklearn's version is just shifted up by the smoothing.

### 4.5 Top-weighted terms for sample items

In [22]:
def top_terms(title, matrix, vocab, k=6):
    """Non-zero TF-IDF weights of one movie, highest first."""
    row = matrix[title_to_idx[title]].toarray().ravel()
    order = np.argsort(-row)
    s = pd.Series(row[order], index=vocab[order])
    return s[s > 0].head(k).round(3)

for t in ["Toy Story (1995)", "Chinatown (1974)", "Blade Runner (1982)"]:
    print(f"{t} | {movies.loc[title_to_idx[t], 'genres']}")
    print(top_terms(t, tfidf_matrix, genre_vocab).to_string(), "\n")

Toy Story (1995) | Adventure|Animation|Children|Comedy|Fantasy
Animation    0.516
Children     0.505
Fantasy      0.483
Adventure    0.417
Comedy       0.268 

Chinatown (1974) | Crime|Film-Noir|Mystery|Thriller
Film-Noir    0.715
Mystery      0.480
Crime        0.387
Thriller     0.330 

Blade Runner (1982) | Action|Sci-Fi|Thriller
Sci-Fi      0.660
Action      0.535
Thriller    0.528 



**Interpretation:** within each movie, the rarest genre gets the highest weight. *Chinatown*'s top term is **Film-Noir** (rare, very distinguishing) and **Thriller** is lowest. For *Toy Story*, **Animation** and **Children** outweigh **Comedy** and **Adventure**, which are in thousands of movies. That is exactly what we want: the weights highlight what makes a movie *specific*, not generic labels. No filler terms appear because genres are a curated vocabulary.

### 4.6 Richer text: adding user tags from `tags.csv`
Genres alone are coarse: many movies share an identical genre string, which causes large ties in Phase 5. `ml-latest-small` ships `tags.csv`, short free-text labels users attached to movies (e.g. "pixar", "time travel", "dark comedy"). The readme suggests using these as the text field when available.

In [23]:
tags["tag_clean"] = tags["tag"].astype(str).str.lower().str.strip()
print("Most frequent tags:")
display(tags.tag_clean.value_counts().head(12).to_frame("count").T)

Most frequent tags:


tag_clean,in netflix queue,atmospheric,funny,superhero,surreal,thought-provoking,sci-fi,disney,quirky,religion,dark comedy,suspense
count,131,41,24,24,24,24,23,23,22,22,21,21


The most common tag, **"in netflix queue"**, is not a description of the movie at all. It's a user's personal to-watch list. Keeping it would make 131 unrelated movies look similar through the words "netflix" and "queue", so it's removed before vectorizing. (Spotting non-content features like this is part of building good item profiles.)

In [24]:
NON_CONTENT_TAGS = {"in netflix queue"}
tags_content = tags[~tags.tag_clean.isin(NON_CONTENT_TAGS)]
print(f"Removed {len(tags) - len(tags_content)} non-content tag rows")

movie_tags = tags_content.groupby("movieId")["tag_clean"].apply(" ".join)
movies["tags_text"] = movies["movieId"].map(movie_tags).fillna("")
n_tagged = int((movies.tags_text != "").sum())
print(f"Movies with at least one tag: {n_tagged:,} of {len(movies):,} ({n_tagged / len(movies):.1%})")

# min_df=2: a word used for only one movie can never be shared, so it can never raise a similarity
tag_tfidf = TfidfVectorizer(stop_words="english", min_df=2, dtype=np.float32)
tag_matrix = tag_tfidf.fit_transform(movies["tags_text"])
tag_vocab = tag_tfidf.get_feature_names_out()
print("Tag TF-IDF matrix shape:", tag_matrix.shape, "| vocabulary size:", len(tag_vocab))

for t in ["Toy Story (1995)", "Blade Runner (1982)"]:
    print(f"\n{t} - top tag terms:")
    print(top_terms(t, tag_matrix, tag_vocab, k=8).to_string())

Removed 131 non-content tag rows


Movies with at least one tag: 1,463 of 9,742 (15.0%)
Tag TF-IDF matrix shape: (9742, 722) | vocabulary size: 722

Toy Story (1995) - top tag terms:
pixar    0.901
fun      0.433

Blade Runner (1982) - top tag terms:
atmospheric       0.408
androids          0.283
philip            0.283
cyberpunk         0.283
existentialism    0.262
artificial        0.262
future            0.257
intelligence      0.253


**Interpretation:** the tag terms are much more specific than genres. *Toy Story*'s top term is **pixar**, and *Blade Runner* gets **androids**, **cyberpunk**, **philip** (k. dick), **artificial intelligence** and **existentialism**. These are exactly the details that separate movies with the same genre string. (**atmospheric** ranks first for *Blade Runner* only because two users applied that tag to it, so its TF is 2.) The trade-off: tags exist for only 15% of movies and are user-generated, so they are noisy. They are used as an **extra signal on top of genres**, not a replacement (see Phase 5.5).

**✔ Phase 4 deliverable:** TF-IDF matrix shape `(9742, 19)` with vocabulary size 19 for genres (plus a `(9742, 722)` tag TF-IDF matrix), and top-weighted terms for 3 sample items with interpretation.

---
# Phase 5 — Cosine Similarity & Ranking Items

### 5.1 Formula
```
cosine_similarity(A, B) = (A · B) / (‖A‖ · ‖B‖)
```
It is the cosine of the **angle** between two vectors. Dividing by both lengths cancels out magnitude, so only *direction* matters. Two vectors pointing the same way score 1, perpendicular vectors (no shared features) score 0. Since TF-IDF values are never negative, the score is always in [0, 1].

### 5.2 Why cosine instead of the raw dot product?
A raw dot product grows with vector **magnitude**: an item with a longer description (or more genres) has more non-zero entries, so it gets a bigger dot product with *everything*, regardless of real similarity. Cosine normalizes that away. Demo on the unnormalized 0/1 genre vectors:

In [25]:
seed = title_to_idx["Toy Story (1995)"]
dot_raw = G01 @ G01[seed]
cos_raw = cosine_similarity(G01[[seed]], G01).ravel()
n_genres = G01.sum(axis=1)

cmp = movies[["title", "genres"]].assign(n_genres=n_genres, dot=dot_raw, cosine=cos_raw.round(3)).drop(index=seed)
print("Top 5 by RAW DOT PRODUCT:")
display(cmp.sort_values(["dot", "n_genres"], ascending=False).head(5))
print("Top 5 by COSINE:")
display(cmp.sort_values("cosine", ascending=False, kind="stable").head(5))

print("Correlation between #genres and average similarity to all other movies:")
avg_dot = (G01 @ G01.sum(axis=0)) / len(G01)
avg_cos = cosine_similarity(G01, G01.sum(axis=0, keepdims=True)).ravel()
print(f"  dot product: {np.corrcoef(n_genres, avg_dot)[0, 1]:.2f}   (movies with many genres look similar to everything)")
print(f"  cosine     : {np.corrcoef(n_genres, avg_cos)[0, 1]:.2f}")

Top 5 by RAW DOT PRODUCT:


,title,genres,n_genres,dot,cosine
2250,Who Framed Roger Rabbit? (1988),Adventure|Animation|Children|Comedy|Crime|Fantasy|Mystery,7.0,5.0,0.845
5819,Robots (2005),Adventure|Animation|Children|Comedy|Fantasy|Sci-Fi|IMAX,7.0,5.0,0.845
6626,Enchanted (2007),Adventure|Animation|Children|Comedy|Fantasy|Musical|Romance,7.0,5.0,0.845
559,Space Jam (1996),Adventure|Animation|Children|Comedy|Fantasy|Sci-Fi,6.0,5.0,0.913
3194,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Romance,6.0,5.0,0.913


Top 5 by COSINE:


,title,genres,n_genres,dot,cosine
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,5.0,5.0,1.0
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,5.0,5.0,1.0
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,5.0,5.0,1.0
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,5.0,5.0,1.0
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,5.0,5.0,1.0


Correlation between #genres and average similarity to all other movies:
  dot product: 0.63   (movies with many genres look similar to everything)
  cosine     : 0.14


With the raw dot product, *Who Framed Roger Rabbit?* (7 genres) scores exactly the same as an identical-genre movie like *Toy Story 2* (dot = 5 for both), so the dot product can't tell "same kind of movie" from "has lots of genres". Across the catalog, movies with many genres look similar to everything (correlation 0.63). Cosine ranks the exact genre matches first, and the size effect almost disappears (0.14). Note that our TF-IDF rows are already L2-normalized (length 1), so for them the dot product *equals* cosine similarity. That is checked below.

### 5.3 Pairwise cosine similarity matrix across all items

In [26]:
# Passing the small dense (9742 x 19) array makes sklearn produce the dense result directly.
# With float32 the 9742 x 9742 result takes ~380 MB (float64 would take ~760 MB).
X_genre = tfidf_matrix.toarray()

t0 = time.perf_counter()
sim_matrix = cosine_similarity(X_genre)
elapsed = time.perf_counter() - t0
print(f"sim_matrix: shape {sim_matrix.shape}, dtype {sim_matrix.dtype}, "
      f"{sim_matrix.nbytes / 1e6:,.0f} MB, computed in {elapsed:.2f}s")
print("Rows are unit length, so dot product == cosine:",
      np.allclose(X_genre[:50] @ X_genre.T, sim_matrix[:50], atol=1e-6))
print("Diagonal (self-similarity) is 1 for movies with genres:",
      np.allclose(np.diag(sim_matrix)[X_genre.sum(1) > 0], 1, atol=1e-6))

sim_matrix: shape (9742, 9742), dtype float32, 380 MB, computed in 0.36s
Rows are unit length, so dot product == cosine: True
Diagonal (self-similarity) is 1 for movies with genres: True


### 5.4 Top-10 most similar movies to a seed

**The readme's function as written**, run on *Toy Story 2*:

In [27]:
def recommend_similar_readme(movie_title, top_n=10):
    idx = movies[movies["title"] == movie_title].index[0]
    scores = list(enumerate(sim_matrix[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]
    return movies.iloc[[i for i, _ in scores]][["title", "genres"]]

out = recommend_similar_readme("Toy Story 2 (1999)")
display(out)
print("Seed movie appears in its own recommendations:", "Toy Story 2 (1999)" in out.title.values)

,title,genres
1706,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy
2355,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
2809,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3000,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy
3568,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy
6194,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy
6486,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy
6948,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy
7760,Asterix and the Vikings (Astérix et les Vikings) (2006),Adventure|Animation|Children|Comedy|Fantasy
8219,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy


Seed movie appears in its own recommendations: True


**Bug:** `[1:top_n+1]` assumes the seed itself is always ranked first so it can be skipped. With genre vectors, many movies have *identical* vectors and tie at similarity 1.0, and Python's stable sort keeps tied items in their original row order. *Toy Story* (row 0) is ranked before *Toy Story 2*, so the slice drops *Toy Story* and keeps *Toy Story 2* as a recommendation for itself. The fix is to exclude the seed by **index**, not by position. The fixed function also reports how many movies are tied at the top score:

In [28]:
def recommend_similar(movie_title, top_n=10, sim=None):
    """Top-N most similar movies to `movie_title` (the seed is excluded by index, not by position)."""
    if movie_title not in title_to_idx.index:
        raise KeyError(f"'{movie_title}' not found in movies.csv")
    idx = title_to_idx[movie_title]
    scores = (sim_matrix[idx] if sim is None else sim).astype(float).copy()
    scores[idx] = -np.inf                                    # never recommend the seed itself
    top = np.argsort(-scores, kind="stable")[:top_n]
    out = movies.iloc[top][["title", "genres"]].copy()
    out["similarity"] = scores[top].round(3)
    n_tied = int(np.isclose(scores, scores[top[0]]).sum())
    tie_note = f"{n_tied} movies tied at the top score" if n_tied > 1 else "no tie at the top score"
    print(f"Seed: {movie_title} | {movies.loc[idx, 'genres']}  ->  {tie_note} {scores[top[0]]:.3f}")
    return out.reset_index(drop=True)

display(recommend_similar("Toy Story (1995)"))
display(recommend_similar("Blade Runner (1982)"))

Seed: Toy Story (1995) | Adventure|Animation|Children|Comedy|Fantasy  ->  12 movies tied at the top score 1.000


,title,genres,similarity
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
7,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
8,Asterix and the Vikings (Astérix et les Vikings) (2006),Adventure|Animation|Children|Comedy|Fantasy,1.0
9,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0


Seed: Blade Runner (1982) | Action|Sci-Fi|Thriller  ->  42 movies tied at the top score 1.000


,title,genres,similarity
0,Lawnmower Man 2: Beyond Cyberspace (1996),Action|Sci-Fi|Thriller,1.0
1,Screamers (1995),Action|Sci-Fi|Thriller,1.0
2,Johnny Mnemonic (1995),Action|Sci-Fi|Thriller,1.0
3,Virtuosity (1995),Action|Sci-Fi|Thriller,1.0
4,Timecop (1994),Action|Sci-Fi|Thriller,1.0
5,Solo (1996),Action|Sci-Fi|Thriller,1.0
6,"Arrival, The (1996)",Action|Sci-Fi|Thriller,1.0
7,"Terminator, The (1984)",Action|Sci-Fi|Thriller,1.0
8,Godzilla (1998),Action|Sci-Fi|Thriller,1.0
9,"Matrix, The (1999)",Action|Sci-Fi|Thriller,1.0


**Sanity check (genres only):** the results make sense. Every recommendation for *Toy Story* has the exact genre combination Adventure|Animation|Children|Comedy|Fantasy (*Antz*, *Toy Story 2*, *The Emperor's New Groove*, *Monsters, Inc.*, ...), and every recommendation for *Blade Runner* is an Action|Sci-Fi|Thriller film. But the ranking is limited: **12 movies tie at exactly 1.0 for *Toy Story* and 42 for *Blade Runner***, so the top 10 is just *whichever tied movies come first in the file*. Genres can say "this is a sci-fi thriller" but can't say which sci-fi thriller is closest. A richer feature is needed to break the ties.

### 5.5 Genres + tags: breaking the ties
Combined score for any two movies:

```
similarity = (1 − w) · cosine(genre vectors) + w · cosine(tag vectors),   w = TAG_WEIGHT = 0.3
```

Each block is compared with cosine on its own, then the two scores are blended. Genres (complete for every movie, curated) carry 70% of the score. Tags (sparse, user-generated) add up to 30% as a bonus. A movie with **no tags simply gets 0 for the tag part**: it can still be recommended on genres, but it can never be *penalized* below its genre match. (Concatenating both blocks into one vector before cosine would not have this property: tagged movies whose tags don't overlap would score *lower* than untagged ones.)

In [29]:
TAG_WEIGHT = 0.3

def content_similarity_row(idx, w=TAG_WEIGHT):
    """Combined genre+tag similarity of movie `idx` to every movie."""
    genre_part = cosine_similarity(tfidf_matrix[idx], tfidf_matrix).ravel()
    tag_part = cosine_similarity(tag_matrix[idx], tag_matrix).ravel()      # 0 if either movie has no tags
    return (1 - w) * genre_part + w * tag_part

for t in ["Toy Story (1995)", "Blade Runner (1982)"]:
    display(recommend_similar(t, sim=content_similarity_row(title_to_idx[t])))

Seed: Toy Story (1995) | Adventure|Animation|Children|Comedy|Fantasy  ->  no tie at the top score 0.883


,title,genres,similarity
0,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.883
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.873
2,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fantasy,0.718
3,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,0.700
4,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.700
5,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.700
6,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.700
7,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.700
8,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.700
9,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,0.700


Seed: Blade Runner (1982) | Action|Sci-Fi|Thriller  ->  no tie at the top score 0.820


,title,genres,similarity
0,"Terminator, The (1984)",Action|Sci-Fi|Thriller,0.820
1,Predator (1987),Action|Sci-Fi|Thriller,0.748
2,"Matrix, The (1999)",Action|Sci-Fi|Thriller,0.740
3,Lawnmower Man 2: Beyond Cyberspace (1996),Action|Sci-Fi|Thriller,0.700
4,Screamers (1995),Action|Sci-Fi|Thriller,0.700
5,Johnny Mnemonic (1995),Action|Sci-Fi|Thriller,0.700
6,Virtuosity (1995),Action|Sci-Fi|Thriller,0.700
7,Timecop (1994),Action|Sci-Fi|Thriller,0.700
8,Solo (1996),Action|Sci-Fi|Thriller,0.700
9,"Arrival, The (1996)",Action|Sci-Fi|Thriller,0.700


**Sanity check (genres + tags):** the top of each list is now decided by shared tags instead of file order. For *Toy Story*, *A Bug's Life* and *Toy Story 2* move to the top (both tagged "pixar"), then *The Lego Movie* (shares "fun"). For *Blade Runner*, *The Terminator* comes first (shares "artificial intelligence", "robots", "sci-fi"), then *Predator* and *The Matrix* (share "sci-fi"). Below that, the remaining places are still ties at 0.700: those movies match on genres but have no tags in common, often no tags at all. For example, *Monsters, Inc.* has no tags in this small dataset, so it can't benefit. That is the honest limit of user-generated tags: they only help the 15% of movies that have them.

### 5.6 The item-item rule this implements
**"Users who liked X also tend to like things similar to X."** Here "similar" is defined entirely by the items' own features (genres, tags). This uses only item features: no other user's behaviour is involved at any point. (Collaborative filtering's "people who liked X also liked Y" uses behaviour instead; that is Task 12.)

**✔ Phase 5 deliverable:** top-10 similar-item lists for 2 seed movies (genres only and genres + tags), with sanity checks, plus the bug fix for the readme function.

In [30]:
# Free the 9742 x 9742 matrix (~380 MB); later sections compute only the rows they need.
del sim_matrix
gc.collect();

---
# Phase 6 — Building User Profiles from Item Features

### 6.1 The approach
Represent a **user** as a vector in the *same* feature space as the movies: the (weighted) average of the feature vectors of every movie they rated highly. If a user mostly loved animated family films, their profile points strongly in the Animation/Children direction. Then recommend the unseen movies whose vectors point in the most similar direction (cosine). In this phase the profile uses the genre TF-IDF vectors, which keeps it easy to read. The capstone adds tags.

### 6.2 Choosing two users with clearly different taste
Rather than picking user IDs at random, choose them by a transparent rule: among users with 20–200 liked movies, the one whose liked movies have the biggest *Comedy minus Action* share, and the one with the biggest *Action minus Comedy* share.

In [31]:
liked = ratings[ratings.rating >= LIKE_THRESHOLD].merge(movies[["movieId", "genres"]], on="movieId")
n_liked = liked.groupby("userId").size()
genre_share = (liked.assign(genre=liked.genres.str.split("|")).explode("genre")
               .groupby(["userId", "genre"]).size().unstack(fill_value=0)
               .div(n_liked, axis=0))       # share of a user's liked movies that carry each genre

eligible = n_liked[n_liked.between(20, 200)].index
comedy_user = int((genre_share.loc[eligible, "Comedy"] - genre_share.loc[eligible, "Action"]).idxmax())
action_user = int((genre_share.loc[eligible, "Action"] - genre_share.loc[eligible, "Comedy"]).idxmax())

for uid in [comedy_user, action_user]:
    print(f"User {uid}: {n_liked[uid]} liked movies | "
          f"Comedy {genre_share.loc[uid, 'Comedy']:.0%}, Action {genre_share.loc[uid, 'Action']:.0%}, "
          f"Drama {genre_share.loc[uid, 'Drama']:.0%}, Thriller {genre_share.loc[uid, 'Thriller']:.0%}")

User 564: 69 liked movies | Comedy 97%, Action 17%, Drama 17%, Thriller 6%
User 164: 26 liked movies | Comedy 4%, Action 96%, Drama 27%, Thriller 35%


### 6.3 Building the profile (readme code, with one fix)
The readme's function has an alignment bug: it filters `idxs` (dropping movies missing from the index) but then takes `weights = rating.values[:len(idxs)]`, which just cuts the *end* off the ratings array. If a movie in the middle is skipped, every later weight shifts onto the wrong movie. The fix filters the DataFrame first, so movie IDs and weights stay paired:

In [32]:
def build_user_profile(user_id, ratings, item_vectors, movie_index, threshold=LIKE_THRESHOLD, weighted=True):
    """Weighted average of the vectors of movies the user rated >= threshold. None if there are none."""
    user_ratings = ratings[(ratings.userId == user_id) & (ratings.rating >= threshold)]
    user_ratings = user_ratings[user_ratings.movieId.isin(movie_index.index)]   # keep ids and weights aligned
    if user_ratings.empty:
        return None
    idxs = movie_index[user_ratings.movieId].to_numpy()
    weights = user_ratings.rating.to_numpy() if weighted else None
    return np.average(item_vectors[idxs].toarray(), axis=0, weights=weights)

def show_profile(profile, vocab=genre_vocab, k=8):
    s = pd.Series(profile, index=vocab).sort_values(ascending=False)
    return s[s > 0].head(k).round(3)

profile_plain = build_user_profile(comedy_user, ratings, tfidf_matrix, movieid_to_idx, weighted=False)
profile_weighted = build_user_profile(comedy_user, ratings, tfidf_matrix, movieid_to_idx, weighted=True)

print(f"User {comedy_user} - profile vector length {profile_weighted.shape[0]} (one weight per genre)")
display(pd.DataFrame({"plain mean": show_profile(profile_plain), "rating-weighted": show_profile(profile_weighted)}))
print("Cosine(plain, weighted) =", round(float(cosine_similarity([profile_plain], [profile_weighted])[0, 0]), 5))

User 564 - profile vector length 19 (one weight per genre)


,plain mean,rating-weighted
Action,0.115,0.112
Adventure,0.087,0.082
Comedy,0.671,0.674
Crime,0.083,0.085
Drama,0.095,0.094
Romance,0.199,0.201
Sci-Fi,0.044,0.042
Thriller,0.032,0.030


Cosine(plain, weighted) = 0.99996


**Plain vs. rating-weighted:** the two profiles are almost identical (cosine ≈ 1). That is expected, because only ratings ≥ 4 are included, so the weights only range from 4.0 to 5.0. A 5★ movie counts just 1.25× as much as a 4★ one. Weighting would matter much more if the weights were centred on the user's own average rating (e.g. `rating − user_mean`), which turns "liked a bit" vs. "loved" into a much bigger difference.

### 6.4 Recommending: profile vs. every unrated movie

In [33]:
def recommend_from_profile(user_id, profile, item_vectors, top_n=10):
    """Cosine between the user profile and every movie; movies the user already rated are excluded."""
    scores = cosine_similarity(profile.reshape(1, -1), item_vectors).ravel()
    seen = movieid_to_idx[ratings.loc[ratings.userId == user_id, "movieId"]].to_numpy()
    scores[seen] = -np.inf
    top = np.argsort(-scores, kind="stable")[:top_n]
    out = movies.iloc[top][["title", "genres"]].copy()
    out["score"] = scores[top].round(3)
    return out.reset_index(drop=True)

recs_plain = recommend_from_profile(comedy_user, profile_plain, tfidf_matrix)
recs_comedy = recommend_from_profile(comedy_user, profile_weighted, tfidf_matrix)
print(f"User {comedy_user} - top 10 (rating-weighted profile):")
display(recs_comedy)
print("Overlap between plain-mean and weighted top-10:", len(set(recs_plain.title) & set(recs_comedy.title)), "/ 10")

User 564 - top 10 (rating-weighted profile):


,title,genres,score
0,Father of the Bride Part II (1995),Comedy,0.923
1,Four Rooms (1995),Comedy,0.923
2,Ace Ventura: When Nature Calls (1995),Comedy,0.923
3,Bio-Dome (1996),Comedy,0.923
4,Friday (1995),Comedy,0.923
5,Black Sheep (1996),Comedy,0.923
6,Mr. Wrong (1996),Comedy,0.923
7,"Steal Big, Steal Little (1995)",Comedy,0.923
8,Flirting With Disaster (1996),Comedy,0.923
9,Down Periscope (1996),Comedy,0.923


Overlap between plain-mean and weighted top-10: 10 / 10


### 6.5 A second user with different taste

In [34]:
profile_action = build_user_profile(action_user, ratings, tfidf_matrix, movieid_to_idx, weighted=True)
print(f"User {action_user} - profile:")
display(show_profile(profile_action).to_frame("weight").T)
recs_action = recommend_from_profile(action_user, profile_action, tfidf_matrix)
print(f"User {action_user} - top 10:")
display(recs_action)

both = pd.concat({f"User {comedy_user} (comedy fan)": recs_comedy[["title", "genres"]],
                  f"User {action_user} (action fan)": recs_action[["title", "genres"]]}, axis=1)
print("Side by side:")
display(both)
print("Movies recommended to both users:", len(set(recs_comedy.title) & set(recs_action.title)))
print("Cosine similarity between the two profiles:",
      round(float(cosine_similarity([profile_weighted], [profile_action])[0, 0]), 3))

User 164 - profile:


,Action,Adventure,Sci-Fi,Thriller,Drama,War,Romance,Western
weight,0.538,0.308,0.287,0.188,0.107,0.06,0.051,0.032


User 164 - top 10:


,title,genres,score
0,"Day After Tomorrow, The (2004)",Action|Adventure|Drama|Sci-Fi|Thriller,0.905
1,"Sound of Thunder, A (2005)",Action|Adventure|Drama|Sci-Fi|Thriller,0.905
2,Children of Men (2006),Action|Adventure|Drama|Sci-Fi|Thriller,0.905
3,Jumper (2008),Action|Adventure|Drama|Sci-Fi|Thriller,0.905
4,The Hunger Games (2012),Action|Adventure|Drama|Sci-Fi|Thriller,0.905
5,Jurassic World (2015),Action|Adventure|Drama|Sci-Fi|Thriller,0.905
6,Jurassic World: Fallen Kingdom (2018),Action|Adventure|Drama|Sci-Fi|Thriller,0.905
7,Escape from L.A. (1996),Action|Adventure|Sci-Fi|Thriller,0.902
8,"Abyss, The (1989)",Action|Adventure|Sci-Fi|Thriller,0.902
9,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller,0.902


Side by side:


User 564 (comedy fan)                          User 164 (action fan)                                        
                                   title  genres                                  title                                  genres
0     Father of the Bride Part II (1995)  Comedy         Day After Tomorrow, The (2004)  Action|Adventure|Drama|Sci-Fi|Thriller
1                      Four Rooms (1995)  Comedy             Sound of Thunder, A (2005)  Action|Adventure|Drama|Sci-Fi|Thriller
2  Ace Ventura: When Nature Calls (1995)  Comedy                 Children of Men (2006)  Action|Adventure|Drama|Sci-Fi|Thriller
3                        Bio-Dome (1996)  Comedy                          Jumper (2008)  Action|Adventure|Drama|Sci-Fi|Thriller
4                          Friday (1995)  Comedy                The Hunger Games (2012)  Action|Adventure|Drama|Sci-Fi|Thriller
5                     Black Sheep (1996)  Comedy                  Jurassic World (2015)  Action|Adventure|Drama|Sci-Fi|Thriller
6                       Mr. Wrong (1996)  Comedy  Jurassic World: Fallen Kingdom (2018)  Action|Adventure|Drama|Sci-Fi|Thriller
7         Steal Big, Steal Little (1995)  Comedy                Escape from L.A. (1996)        Action|Adventure|Sci-Fi|Thriller
8          Flirting With Disaster (1996)  Comedy                      Abyss, The (1989)        Action|Adventure|Sci-Fi|Thriller
9                  Down Periscope (1996)  Comedy            Escape from New York (1981)        Action|Adventure|Sci-Fi|Thriller

Movies recommended to both users: 0
Cosine similarity between the two profiles: 0.261


**Result:** the two users get completely different lists with zero overlap. The comedy fan's profile is dominated by Comedy (then Romance) and their whole list is comedies. The action fan's profile is dominated by Action, Adventure and Sci-Fi, and their list is action/sci-fi adventures. The two profiles have a cosine similarity of only 0.26. This shows personalization: the *same* algorithm and item vectors give different outputs purely because the user profiles differ.

The genre-only representation still has the tie problem here: all 10 comedies score exactly 0.923 (they are all plain "Comedy"), so which comedies appear is again decided by file order. The capstone uses genres + tags to rank more finely.

**✔ Phase 6 deliverable:** two distinct user profiles with their top-10 recommendations, demonstrating personalization.

---
# Phase 7 — Strengths & Weaknesses of Content-Based Filtering

| | Point | Why |
|---|---|---|
| ✅ Strength | **No item cold start** | A brand-new movie is recommendable immediately from its own features; it needs no ratings |
| ✅ Strength | **No other users' data needed** | Works for one user alone; no need for a big user base, and no privacy dependency on others |
| ✅ Strength | **Explainable** | "Because you liked *Toy Story*": each recommendation can be traced to a liked item and shared features |
| ✅ Strength | **Niche-friendly** | Can recommend obscure items that nobody else rated, as long as their features match |
| ❌ Weakness | **Limited serendipity** | Only recommends things similar to what the user already likes; can never discover a genuinely new taste |
| ❌ Weakness | **Capped by feature quality** | Can only be as good as the item features. Genres alone can't separate a great sci-fi film from a bad one |
| ❌ Weakness | **User cold start remains** | A new user with no history has no profile, so nothing to compare against (Exercise 4) |
| ❌ Weakness | **No notion of quality or popularity** | Similarity ≠ goodness: a terrible movie with the right genres scores as high as a classic |

Two quick demonstrations from the data.

**Item cold start, solved:** a hypothetical new movie, added with only its genres and zero ratings, can be matched immediately:

In [35]:
new_movie_vec = tfidf.transform(["Animation|Children|Comedy"])      # a brand-new movie, zero ratings
scores = cosine_similarity(new_movie_vec, tfidf_matrix).ravel()
print("New movie (genres Animation|Children|Comedy) - the most similar existing movies:")
display(movies.iloc[np.argsort(-scores, kind="stable")[:5]][["title", "genres"]])

# ...and it can go straight into a user's recommendations
print(f"Its similarity to user {comedy_user}'s profile: {cosine_similarity(new_movie_vec, [profile_weighted])[0, 0]:.3f}")
print(f"Its similarity to user {action_user}'s profile : {cosine_similarity(new_movie_vec, [profile_action])[0, 0]:.3f}")

New movie (genres Animation|Children|Comedy) - the most similar existing movies:


,title,genres
599,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy
1756,"Rugrats Movie, The (1998)",Animation|Children|Comedy
2806,Chicken Run (2000),Animation|Children|Comedy
2978,Rugrats in Paris: The Movie (2000),Animation|Children|Comedy
5170,Garfield: The Movie (2004),Animation|Children|Comedy


Its similarity to user 564's profile: 0.321
Its similarity to user 164's profile : 0.034


**The filter bubble, measured:** compare the genre variety of what each user liked with what the recommender gives back.

In [36]:
def genre_set(genre_strings):
    return set(genre_strings.str.split("|").explode()) - {"(no genres listed)"}

rows = []
for uid, recs in [(comedy_user, recs_comedy), (action_user, recs_action)]:
    hist = liked.loc[liked.userId == uid, "genres"]
    top_genre = genre_share.loc[uid].idxmax()
    rows.append([uid, len(genre_set(hist)), len(genre_set(recs.genres)),
                 ", ".join(sorted(genre_set(hist) - genre_set(recs.genres))),
                 f"{top_genre}: {hist.str.contains(top_genre, regex=False).mean():.0%} of liked -> "
                 f"{recs.genres.str.contains(top_genre, regex=False).mean():.0%} of recs"])
pd.DataFrame(rows, columns=["user", "genres in liked history", "genres in top-10 recs",
                            "liked genres never recommended", "top genre"])

,user,genres in liked history,genres in top-10 recs,liked genres never recommended,top genre
0,564,11,1,"Action, Adventure, Crime, Drama, Fantasy, Mystery, Romance, Sci-Fi...",Comedy: 97% of liked -> 100% of recs
1,164,13,5,"Children, Comedy, Crime, Fantasy, Horror, Romance, War, Western",Action: 96% of liked -> 100% of recs


### Filter bubble, in my own words
A purely content-based recommender only ever shows a user more of what they already liked. The table shows it: the comedy fan liked movies across 11 genres but gets recommendations from just 1 (pure Comedy), and the action fan's 13 genres shrink to 5. The dominant genre goes from ~96% of their history to 100% of their recommendations. If the user then watches and likes those recommendations, the next profile becomes even more concentrated, and the next round of recommendations narrower still. Over time this becomes a self-reinforcing loop. The user is never shown a documentary, a western or a foreign film they might have loved, simply because nothing in their past "points" there. The system isn't wrong about what they like; it just never gives them the chance to like anything new.

### Next step (preview only, not built here)
**Collaborative filtering (Task 12)** recommends based on what *similar users* liked, rather than on item features. If people with your taste also loved a western, it can recommend that western even though nothing in your own history looks like it. That breaks the bubble and brings in quality signals, but in turn it suffers from item cold start, which content-based handles.

**✔ Phase 7 deliverable:** strengths/weaknesses table and filter-bubble explanation (with a measured demonstration).

---
# Phase 8 — Capstone: End-to-End Content-Based Movie Recommender

Phases 3–6 combined into one reusable pipeline:

```
load data → item profiles (genre TF-IDF [+ scaled year] and tag TF-IDF) → item-item similarity matrix
          → user profile on demand → score every unseen movie → top-N with explanations
```

**Scoring** (same formula for item↔item and user↔item):
`score = (1 − tag_weight) · cosine(genre part) + tag_weight · cosine(tag part)`

**Parameters** (the defaults are the model; the others exist so the exercises can reuse the same pipeline):
- `like_threshold=4.0`: ratings counted as "liked" when building the profile
- `tag_weight=0.3`: share of the score from tags (Phase 5.5)
- `year_weight=0.0`: weight of the scaled release year appended to the genre vector (Exercise 1)
- `use_idf=True`: set `False` for the TF-only ablation (Exercise 5)
- `precompute_similarity=True`: build the full item-item matrix once, or compute rows on demand (scaling note, step 5)

In [37]:
class ContentBasedRecommender:
    """Content-based movie recommender: genres (+ year) and tags -> TF-IDF -> cosine similarity."""

    def __init__(self, data_dir, like_threshold=4.0, tag_weight=0.3, year_weight=0.0,
                 use_idf=True, precompute_similarity=True):
        self.data_dir = Path(data_dir)
        self.like_threshold = like_threshold
        self.tag_weight = tag_weight
        self.year_weight = year_weight
        self.use_idf = use_idf
        self.precompute_similarity = precompute_similarity

    # ---------- offline part: build everything that doesn't depend on the request ----------
    def fit(self):
        t = {}
        t0 = time.perf_counter()
        self.movies = pd.read_csv(self.data_dir / "movies.csv").reset_index(drop=True)
        self.ratings = pd.read_csv(self.data_dir / "ratings.csv")
        tags = pd.read_csv(self.data_dir / "tags.csv")
        t["1. load data"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        self.movieid_to_idx = pd.Series(self.movies.index, index=self.movies.movieId)
        titles = pd.Series(self.movies.index, index=self.movies.title)
        self.title_to_idx = titles[~titles.index.duplicated(keep="first")]

        # Genre block: TF-IDF over whole-genre tokens, optionally + scaled release year
        self.genre_vec = TfidfVectorizer(tokenizer=genre_tokenizer, token_pattern=None, lowercase=False,
                                         use_idf=self.use_idf, dtype=np.float32)
        G = self.genre_vec.fit_transform(self.movies.genres).toarray()
        if self.year_weight > 0:
            year = self.movies.title.str.extract(r"\((\d{4})\)\s*$", expand=False).astype(float)
            year = MinMaxScaler().fit_transform(year.fillna(year.median()).to_frame()).astype(np.float32)
            G = np.hstack([G, self.year_weight * year])
        self.G = G                                                   # dense, 9742 x ~20: tiny

        # Tag block: TF-IDF over the concatenated tags of each movie (sparse, mostly empty rows)
        tags["tag"] = tags["tag"].astype(str).str.lower().str.strip()
        tags = tags[~tags["tag"].isin(NON_CONTENT_TAGS)]             # e.g. "in netflix queue" (Phase 4.6)
        tag_text = self.movies.movieId.map(tags.groupby("movieId")["tag"].apply(" ".join)).fillna("")
        self.tag_vec = TfidfVectorizer(stop_words="english", min_df=2, use_idf=self.use_idf, dtype=np.float32)
        self.T = self.tag_vec.fit_transform(tag_text)
        t["2. item profiles"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        self.item_sim = self._similarity(self.G, self.T) if self.precompute_similarity else None
        t["3. item-item similarity"] = time.perf_counter() - t0

        t0 = time.perf_counter()
        self.liked_by_user = {uid: df for uid, df in
                              self.ratings[self.ratings.rating >= self.like_threshold].groupby("userId")}
        self.seen_by_user = {uid: self.movieid_to_idx[df.movieId].to_numpy()
                             for uid, df in self.ratings.groupby("userId")}
        positives = self.ratings[self.ratings.rating >= self.like_threshold].groupby("movieId").size()
        self.popular_idx = self.movieid_to_idx[positives.sort_values(ascending=False).index].to_numpy()
        t["4. user lookups"] = time.perf_counter() - t0

        self.fit_timings_ = pd.Series(t).round(3)
        return self

    def _similarity(self, g_rows, t_rows):
        """(1 - w) * cos(genre part) + w * cos(tag part), for the given rows against all movies."""
        sim = cosine_similarity(g_rows, self.G)
        sim *= (1 - self.tag_weight)
        tag_sim = cosine_similarity(t_rows, self.T, dense_output=False)
        if sparse.issparse(tag_sim):              # add only the non-zero tag similarities (saves memory)
            tag_sim = tag_sim.tocoo()
            sim[tag_sim.row, tag_sim.col] += self.tag_weight * tag_sim.data
        else:
            sim += self.tag_weight * tag_sim
        return sim.astype(np.float32, copy=False)

    def _item_rows(self, idxs):
        idxs = np.atleast_1d(idxs)
        if self.item_sim is not None:
            return self.item_sim[idxs]
        return self._similarity(self.G[idxs], self.T[idxs])

    # ---------- online part: what happens per request ----------
    def build_user_profile(self, user_id):
        """Rating-weighted mean of the liked movies' genre and tag vectors. None if no liked movies."""
        user_liked = self.liked_by_user.get(user_id)
        if user_liked is None or user_liked.empty:
            return None
        idxs = self.movieid_to_idx[user_liked.movieId].to_numpy()
        w = user_liked.rating.to_numpy()
        g_profile = np.average(self.G[idxs], axis=0, weights=w).reshape(1, -1)
        t_profile = np.asarray(self.T[idxs].multiply(w[:, None]).sum(axis=0) / w.sum())
        return g_profile, t_profile, idxs

    def recommend_for_user(self, user_id, top_n=10, explain=True):
        profile = self.build_user_profile(user_id)
        if profile is None:                                          # user cold start (Exercise 4)
            seen = set(self.seen_by_user.get(user_id, []))
            top = [i for i in self.popular_idx if i not in seen][:top_n]
            out = self.movies.iloc[top][["title", "genres"]].copy()
            out["score"] = np.nan
            out["because_you_liked"] = "cold start: no liked movies -> most-liked movies overall (not content-based)"
            return out.reset_index(drop=True)

        g_profile, t_profile, liked_idx = profile
        scores = self._similarity(g_profile, t_profile).ravel().astype(float)
        scores[self.seen_by_user[user_id]] = -np.inf                  # exclude everything already rated
        top = np.argsort(-scores, kind="stable")[:top_n]

        out = self.movies.iloc[top][["title", "genres"]].copy()
        out["score"] = scores[top].round(3)
        if explain:
            sims = self._item_rows(top)[:, liked_idx]                 # each rec vs each liked movie
            best = sims.argmax(axis=1)
            out["because_you_liked"] = self.movies.title.to_numpy()[liked_idx[best]]
            out["similarity_to_it"] = sims[np.arange(len(top)), best].round(3)
        return out.reset_index(drop=True)

    def more_like_this(self, title, top_n=10):
        """Item-item recommendation for a single movie; no user needed (Exercise 3)."""
        idx = self.title_to_idx[title]
        scores = self._item_rows(idx).ravel().astype(float)
        scores[idx] = -np.inf
        top = np.argsort(-scores, kind="stable")[:top_n]
        out = self.movies.iloc[top][["title", "genres"]].copy()
        out["similarity"] = scores[top].round(3)
        return out.reset_index(drop=True)

    def user_history(self, user_id, k=10):
        """The user's highest-rated movies, to judge the recommendations against."""
        r = self.ratings[self.ratings.userId == user_id].merge(self.movies, on="movieId")
        return r.sort_values(["rating", "timestamp"], ascending=False).head(k)[["title", "genres", "rating"]]

In [38]:
t0 = time.perf_counter()
recommender = ContentBasedRecommender(DATA_DIR).fit()
total_fit = time.perf_counter() - t0
print(f"Fit time breakdown (s):\n{recommender.fit_timings_.to_string()}\nTotal fit: {total_fit:.2f}s")
print(f"\nGenre block: {recommender.G.shape} | Tag block: {recommender.T.shape} | "
      f"item-item matrix: {recommender.item_sim.shape}, {recommender.item_sim.nbytes / 1e6:,.0f} MB")


def recommend_for_user(user_id, top_n=10):
    """Readme-required entry point: top-N recommended titles for a user, excluding movies already rated."""
    return recommender.recommend_for_user(user_id, top_n=top_n)

Fit time breakdown (s):
1. load data               0.072
2. item profiles           0.117
3. item-item similarity    0.471
4. user lookups            0.237
Total fit: 0.90s

Genre block: (9742, 19) | Tag block: (9742, 722) | item-item matrix: (9742, 9742), 380 MB


### 8.1 Testing on 3 different users
Users: the comedy fan and the action fan from Phase 6, plus a third chosen by the same rule for a different taste: the eligible user with the highest *Horror* share.

In [39]:
horror_user = int(genre_share.loc[eligible, "Horror"].idxmax())
test_users = {comedy_user: "comedy fan", action_user: "action fan", horror_user: "horror fan"}

def evaluate_user(uid, label):
    print("=" * 110)
    print(f"USER {uid} ({label}) - {n_liked[uid]} liked movies, top genres in liked history: "
          + ", ".join(f"{g} {s:.0%}" for g, s in genre_share.loc[uid].sort_values(ascending=False).head(4).items()))
    print("Highest-rated movies in history:")
    display(recommender.user_history(uid, k=6))
    print("Recommendations:")
    display(recommend_for_user(uid, top_n=10))

for uid, label in test_users.items():
    evaluate_user(uid, label)

USER 564 (comedy fan) - 69 liked movies, top genres in liked history: Comedy 97%, Romance 26%, Action 17%, Drama 17%
Highest-rated movies in history:


,title,genres,rating
138,The Wedding Ringer (2015),Comedy,5.0
57,Drillbit Taylor (2008),Comedy,5.0
70,Adventureland (2009),Comedy|Drama,5.0
134,Horrible Bosses 2 (2014),Comedy|Crime,5.0
32,Mean Girls (2004),Comedy,5.0
109,"Dictator, The (2012)",Comedy,5.0


Recommendations:


,title,genres,score,because_you_liked,similarity_to_it
0,Zoolander (2001),Comedy,0.780,Old School (2003),0.855
1,Airheads (1994),Comedy,0.765,Old School (2003),0.882
2,The DUFF (2015),Comedy,0.761,Mean Girls (2004),0.915
3,Meet the Fockers (2004),Comedy,0.726,Meet the Parents (2000),0.891
4,Heathers (1989),Comedy,0.720,Mean Girls (2004),0.959
5,Ferris Bueller's Day Off (1986),Comedy,0.720,Mean Girls (2004),0.959
6,Napoleon Dynamite (2004),Comedy,0.720,Mean Girls (2004),0.959
7,Monty Python's The Meaning of Life (1983),Comedy,0.717,Old School (2003),0.810
8,Billy Madison (1995),Comedy,0.669,Mean Girls (2004),0.780
9,Losin' It (1983),Comedy,0.667,Superbad (2007),0.836


USER 164 (action fan) - 26 liked movies, top genres in liked history: Action 96%, Adventure 54%, Sci-Fi 46%, Thriller 35%
Highest-rated movies in history:


...

,title,genres,score,because_you_liked,similarity_to_it
0,"Arrival, The (1996)",Action|Sci-Fi|Thriller,0.737,Independence Day (a.k.a. ID4) (1996),0.898
1,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.735,Star Wars: Episode V - The Empire Strikes Back (1980),0.863
2,Star Wars: Episode III - Revenge of the Sith (2005),Action|Adventure|Sci-Fi,0.694,Star Wars: Episode V - The Empire Strikes Back (1980),0.867
3,Terminator Salvation (2009),Action|Adventure|Sci-Fi|Thriller,0.674,Jurassic Park (1993),0.700
4,Spider-Man (2002),Action|Adventure|Sci-Fi|Thriller,0.671,Superman (1978),0.925
5,Serenity (2005),Action|Adventure|Sci-Fi,0.666,Star Wars: Episode V - The Empire Strikes Back (1980),0.700
6,Stargate (1994),Action|Adventure|Sci-Fi,0.659,Star Wars: Episode V - The Empire Strikes Back (1980),0.700
7,X2: X-Men United (2003),Action|Adventure|Sci-Fi|Thriller,0.656,Superman (1978),0.809
8,Superman III (1983),Action|Adventure|Sci-Fi,0.656,Superman (1978),1.000
9,Green Lantern (2011),Action|Adventure|Sci-Fi,0.646,Superman (1978),0.768


USER 571 (horror fan) - 23 liked movies, top genres in liked history: Horror 87%, Thriller 48%, Comedy 22%, Drama 22%
Highest-rated movies in history:


,title,genres,rating
42,Plan 9 from Outer Space (1959),Horror|Sci-Fi,5.0
77,Pet Sematary (1989),Horror,5.0
23,"Amityville Horror, The (1979)",Drama|Horror|Mystery|Thriller,5.0
3,Interview with the Vampire: The Vampire Chronicles (1994),Drama|Horror,5.0
9,Night of the Living Dead (1968),Horror|Sci-Fi|Thriller,5.0
38,Scream (1996),Comedy|Horror|Mystery|Thriller,5.0


Recommendations:


,title,genres,score,because_you_liked,similarity_to_it
0,Cujo (1983),Horror|Thriller,0.732,Misery (1990),0.778
1,Fido (2006),Comedy|Horror|Thriller,0.691,Night of the Living Dead (1968),0.801
2,My Demon Lover (1987),Comedy|Horror,0.685,"Omen, The (1976)",0.648
3,Shaun of the Dead (2004),Comedy|Horror,0.657,Night of the Living Dead (1968),0.671
4,Session 9 (2001),Horror|Thriller,0.656,"Birds, The (1963)",0.700
5,Scream 3 (2000),Comedy|Horror|Mystery|Thriller,0.653,Scream (1996),1.000
6,"Babadook, The (2014)",Drama|Horror|Thriller,0.645,Rosemary's Baby (1968),0.754
7,Rose Red (2002),Horror|Mystery|Thriller,0.631,"Omen, The (1976)",0.700
8,Bug (2007),Drama|Horror|Thriller,0.630,Rosemary's Baby (1968),0.798
9,Relative Fear (1994),Horror|Thriller,0.629,"Birds, The (1963)",0.700


### 8.2 Manual evaluation
- **Comedy fan (user 564):** 97% of their liked movies are comedies (*Mean Girls*, *Old School*, *Superbad*, *Meet the Parents*). All 10 recommendations are comedies, and the explanations make sense: *Zoolander* because they liked *Old School* (both tagged "will ferrell"), *Meet the Fockers* because they liked *Meet the Parents* ("robert de niro"), and *Heathers*, *Ferris Bueller* and *Napoleon Dynamite* because they liked *Mean Girls* ("high school"). **Reasonable**, though very "safe": they also liked some romances and none are recommended.
- **Action fan (user 164):** liked *Independence Day*, *Predator*, *Superman*, *Star Wars: Episode V*. Recommendations: *Star Wars IV* and *III* (because of *Episode V*), *Spider-Man*, *X2* and *Superman III* (because of *Superman*; *Spider-Man* shares its "superhero" tag), and *The Arrival* (an alien-invasion film, because of *Independence Day*; both tagged "aliens"). **Very reasonable**.
- **Horror fan (user 571):** liked *Pet Sematary*, *Night of the Living Dead*, *Scream*, *Misery*, *Rosemary's Baby*. Recommendations: *Cujo* (Stephen King, like *Misery*), *Scream 3* (because of *Scream*), zombie comedies *Shaun of the Dead* and *Fido* (because of *Night of the Living Dead*, tagged "zombies"), *The Babadook* (because of *Rosemary's Baby*). **Reasonable**. Three users, three completely different lists from the same model.
- **Limitations visible in the results:** quality isn't considered. *Superman III* and *Green Lantern* are recommended to the action fan with high confidence purely because they are similar to *Superman*, not because they are good. Some explanations are weak too: *Terminator Salvation* "because you liked *Jurassic Park*" at similarity 0.70 is just a genre match.

### 8.3 The "explain this recommendation" feature
The `because_you_liked` / `similarity_to_it` columns above come from step 4 of the readme: for each recommended movie, the item-item similarity to every movie the user liked is looked up, and the most similar one is reported. For example:

In [40]:
ex = recommend_for_user(action_user, top_n=5)
for _, r in ex.iterrows():
    print(f"• {r.title}  ->  because you liked {r.because_you_liked}  (similarity {r.similarity_to_it:.2f})")

• Arrival, The (1996)  ->  because you liked Independence Day (a.k.a. ID4) (1996)  (similarity 0.90)
• Star Wars: Episode IV - A New Hope (1977)  ->  because you liked Star Wars: Episode V - The Empire Strikes Back (1980)  (similarity 0.86)
• Star Wars: Episode III - Revenge of the Sith (2005)  ->  because you liked Star Wars: Episode V - The Empire Strikes Back (1980)  (similarity 0.87)
• Terminator Salvation (2009)  ->  because you liked Jurassic Park (1993)  (similarity 0.70)
• Spider-Man (2002)  ->  because you liked Superman (1978)  (similarity 0.93)


### 8.4 Timing the full pipeline & scaling bottlenecks

In [41]:
all_users = sorted(recommender.ratings.userId.unique())

t0 = time.perf_counter()
for uid in all_users:
    recommender.recommend_for_user(uid, top_n=10, explain=True)
t_all_pre = time.perf_counter() - t0

# Same thing without the precomputed item-item matrix (rows computed per request instead)
t0 = time.perf_counter()
rec_ondemand = ContentBasedRecommender(DATA_DIR, precompute_similarity=False).fit()
t_fit_od = time.perf_counter() - t0
t0 = time.perf_counter()
for uid in all_users:
    rec_ondemand.recommend_for_user(uid, top_n=10, explain=True)
t_all_od = time.perf_counter() - t0

assert all(recommender.recommend_for_user(u).title.equals(rec_ondemand.recommend_for_user(u).title)
           for u in all_users[:25]), "both modes must give identical recommendations"

timing = pd.DataFrame({
    "fit (offline) s": [total_fit, t_fit_od],
    f"recommend all {len(all_users)} users s": [t_all_pre, t_all_od],
    "per request ms": [1000 * t_all_pre / len(all_users), 1000 * t_all_od / len(all_users)],
    "item-item matrix MB": [recommender.item_sim.nbytes / 1e6, 0],
}, index=["precomputed item-item matrix", "on-demand rows"]).round(2)
timing

,fit (offline) s,recommend all 610 users s,per request ms,item-item matrix MB
precomputed item-item matrix,0.90,4.87,7.99,379.63
on-demand rows,0.28,6.29,10.31,0.00


In [42]:
del rec_ondemand
gc.collect()

n_items = len(recommender.movies)
for n_cat in [n_items, 100_000, 1_000_000, 10_000_000]:
    print(f"{n_cat:>12,} items -> full float32 item-item matrix = {n_cat**2 * 4 / 1e9:>12,.1f} GB")

       9,742 items -> full float32 item-item matrix =          0.4 GB
     100,000 items -> full float32 item-item matrix =         40.0 GB
   1,000,000 items -> full float32 item-item matrix =      4,000.0 GB
  10,000,000 items -> full float32 item-item matrix =    400,000.0 GB


### Scaling note: where this breaks at real-world size

On this dataset everything is fast. In my test run the whole pipeline fit in under a second, and a request with explanations took a few milliseconds, faster with the precomputed matrix than with on-demand rows (the table above has this run's exact numbers; they vary from machine to machine). Precomputing the item-item matrix costs ~380 MB of memory and most of the fit time, and in exchange the explanation look-ups get cheaper. Both modes give identical recommendations. At real-world scale, the bottlenecks are:

1. **The item-item similarity matrix grows as n².** 9,742 movies need ~0.4 GB, 1 million items would need ~4,000 GB, and 10 million items ~400,000 GB. It can't be stored. Production systems precompute only the **top-K neighbours per item** (e.g. the 100 most similar), or use an **approximate nearest-neighbour index** (FAISS, ScaNN, HNSW) to find similar vectors without comparing against everything.
2. **Scoring every item on every request is O(n·d).** Fine for 10K items, too slow for 100M products. Real systems use a two-stage design: **candidate generation** (cheaply pull a few hundred candidates via ANN search) then **ranking** (score only those carefully).
3. **Rebuilding the user profile on every request.** Here it's a quick average, but with long histories and millions of users it should be **precomputed offline** and updated incrementally when a new rating arrives (a running weighted average), not recomputed from scratch.
4. **Refitting TF-IDF when new items arrive.** New movies can be transformed with the existing vocabulary immediately (that's the item cold-start strength), but IDF weights drift, so the vectorizer is refit periodically offline, not per request.

**✔ Phase 8 deliverable:** working `recommend_for_user()`, tested on 3 users with history, results and explanations, plus the timing and scaling note.

---
# Exercises (Must Do)

A helper used by several exercises: how much do two models' top-10 lists overlap across **all 610 users**?

In [43]:
def overlap_at_k(model_a, model_b, users=all_users, k=10):
    """Average share of top-k titles that two models have in common, over all users."""
    ov = []
    for uid in users:
        a = set(model_a.recommend_for_user(uid, top_n=k, explain=False).title)
        b = set(model_b.recommend_for_user(uid, top_n=k, explain=False).title)
        ov.append(len(a & b) / k)
    ov = np.array(ov)
    return pd.Series({"mean overlap@10": ov.mean(), "users with identical top-10": (ov == 1).mean(),
                      "users with <=5 shared": (ov <= 0.5).mean()}).round(3)

base = ContentBasedRecommender(DATA_DIR, precompute_similarity=False).fit()     # same model as the capstone

## Exercise 1 — Release year as a scaled numeric feature
The min-max scaled year is appended to the genre vector with weight `year_weight`. The genre TF-IDF vector has length 1 and the year is in [0, 1], so `year_weight=1` lets the year count as much as all genres together. Several weights are tested:

In [44]:
rows = {}
for w in [0.25, 0.5, 1.0]:
    with_year = ContentBasedRecommender(DATA_DIR, year_weight=w, precompute_similarity=False).fit()
    rows[f"year_weight={w}"] = overlap_at_k(base, with_year)
display(pd.DataFrame(rows).T)

with_year = ContentBasedRecommender(DATA_DIR, year_weight=0.5, precompute_similarity=False).fit()
cmp = pd.concat({"no year": base.more_like_this("Toy Story (1995)")[["title", "similarity"]],
                 "year_weight=0.5": with_year.more_like_this("Toy Story (1995)")[["title", "similarity"]]}, axis=1)
print("More like 'Toy Story (1995)':")
display(cmp)
print(f"User {comedy_user} - top 10 with year_weight=0.5:")
display(with_year.recommend_for_user(comedy_user, explain=False))

,mean overlap@10,users with identical top-10,users with <=5 shared
year_weight=0.25,0.832,0.220,0.062
year_weight=0.5,0.691,0.046,0.203
year_weight=1.0,0.474,0.011,0.648


More like 'Toy Story (1995)':


no year                                                     year_weight=0.5           
                                            title similarity                                                    title similarity
0                            Bug's Life, A (1998)      0.883                                     Bug's Life, A (1998)      0.896
1                              Toy Story 2 (1999)      0.873                                       Toy Story 2 (1999)      0.873
2                           The Lego Movie (2014)      0.718                                    The Lego Movie (2014)      0.723
3                                     Antz (1998)      0.700                                              Antz (1998)      0.700
4  Adventures of Rocky and Bullwinkle, The (2000)      0.700           Adventures of Rocky and Bullwinkle, The (2000)      0.700
5                Emperor's New Groove, The (2000)      0.700                         Emperor's New Groove, The (2000)      0.700
6                           Monsters, Inc. (2001)      0.700                                    Monsters, Inc. (2001)      0.700
7                                Wild, The (2006)      0.700                                         Wild, The (2006)      0.699
8                          Shrek the Third (2007)      0.700  Asterix and the Vikings (Astérix et les Vikings) (2006)      0.699
9                  Tale of Despereaux, The (2008)      0.700                                   Shrek the Third (2007)      0.699

User 564 - top 10 with year_weight=0.5:


,title,genres,score
0,Zoolander (2001),Comedy,0.783
1,The DUFF (2015),Comedy,0.770
2,Airheads (1994),Comedy,0.765
3,Meet the Fockers (2004),Comedy,0.731
4,Napoleon Dynamite (2004),Comedy,0.725
5,Heathers (1989),Comedy,0.717
6,Ferris Bueller's Day Off (1986),Comedy,0.715
7,Monty Python's The Meaning of Life (1983),Comedy,0.711
8,Chalet Girl (2011),Comedy|Romance,0.681
9,Billy Madison (1995),Comedy,0.670


**Finding:** yes, the year changes the recommendations meaningfully, and how much depends on its weight:
- **0.25:** users keep 83% of their top-10 on average. The year mostly acts as a **tie-breaker**.
- **0.5:** 69% kept. In *Toy Story*'s list the tied movies get reordered by release date: *Asterix and the Vikings* (2006) moves above *Shrek the Third* (2007), and *The Tale of Despereaux* (2008) drops out. The comedy fan gets *Chalet Girl* (2011), closer in era to the 2000s–2010s comedies they liked.
- **1.0:** only 47% kept, and for 65% of users at most 5 of the 10 survive. At this weight the year counts as much as all genres combined (Phase 3 showed it alone gives *Toy Story* vs. *The Shining* a similarity of 0.19), so era starts competing with the content itself.

So a small year weight is a sensible refinement; a large one swamps the genre signal. (And Phase 3 showed the extreme case: an *unscaled* year makes every movie look identical.)

## Exercise 2 — Cosine similarity vs. Euclidean distance
**(a) On our L2-normalized TF-IDF vectors**, the two are mathematically tied: for unit vectors, `‖a − b‖² = ‖a‖² + ‖b‖² − 2a·b = 2 − 2·cos(a, b)`. Smaller distance ⇔ larger cosine, so the **rankings are identical**. The helper below checks this for every movie as the seed: it finds the best match(es) under each metric and reports a disagreement when the two sets of best matches share no movie.

In [45]:
def compare_top_results(X, seeds, chunk=1000):
    # For each seed: do the best-cosine and best-Euclidean sets share a movie? (sets, because of ties)
    # Also returns the overlap of the two top-10 lists and Euclidean's top pick.
    disagree, overlap10, euc_pick = [], [], []
    for start in range(0, len(seeds), chunk):
        rows_idx = seeds[start:start + chunk]
        r = np.arange(len(rows_idx))
        cos = cosine_similarity(X[rows_idx], X)
        euc = euclidean_distances(X[rows_idx], X)
        cos[r, rows_idx] = -np.inf                                   # exclude the seed itself
        euc[r, rows_idx] = np.inf
        best_cos = cos >= cos.max(axis=1, keepdims=True) - 1e-9
        best_euc = euc <= euc.min(axis=1, keepdims=True) + 1e-9
        disagree.append(~(best_cos & best_euc).any(axis=1))
        top_c = np.argsort(-cos, axis=1, kind="stable")[:, :10]
        top_e = np.argsort(euc, axis=1, kind="stable")[:, :10]
        overlap10 += [len(set(a) & set(b)) / 10 for a, b in zip(top_c, top_e)]
        euc_pick.append(top_e[:, 0])
    return np.concatenate(disagree), np.array(overlap10), np.concatenate(euc_pick)

Xn = tfidf_matrix.toarray().astype(np.float64)          # unit-length rows (float64 for an exact check)
has_genre = np.where(Xn.sum(axis=1) > 0)[0]              # the 34 genre-less movies have a zero vector, not unit length
cos_all = cosine_similarity(Xn[has_genre[:2000]], Xn[has_genre])
euc_all = euclidean_distances(Xn[has_genre[:2000]], Xn[has_genre])
print("‖a-b‖ == sqrt(2 - 2cos) for 2,000 seeds x all movies:",
      np.allclose(euc_all, np.sqrt(np.clip(2 - 2 * cos_all, 0, None)), atol=1e-6))
del cos_all, euc_all

dis_n, ov_n, _ = compare_top_results(Xn, has_genre)
print(f"Normalized TF-IDF - seeds where the top result differs: {dis_n.sum():,} of {len(has_genre):,}")

‖a-b‖ == sqrt(2 - 2cos) for 2,000 seeds x all movies: True


Normalized TF-IDF - seeds where the top result differs: 0 of 9,708


**(b) On unnormalized 0/1 genre vectors.** Same comparison: for every movie as the seed, do the best match(es) under cosine and under Euclidean distance have a movie in common? (Sets, because of ties.) The top-10 lists are also compared:

In [46]:
dis_g, ov_g, _ = compare_top_results(G01, has_genre)
print(f"Raw 0/1 genres - seeds where the top result differs: {dis_g.sum():,} of {len(has_genre):,} ({dis_g.mean():.1%})")
print(f"Raw 0/1 genres - average overlap of the two top-10 lists: {ov_g.mean():.1%}")
print(f"(Genre combinations: {movies.genres.nunique()} distinct; "
      f"{(movies.genres.map(movies.genres.value_counts()) > 1).mean():.0%} of movies have an exact genre twin)")

Raw 0/1 genres - seeds where the top result differs: 0 of 9,708 (0.0%)
Raw 0/1 genres - average overlap of the two top-10 lists: 95.0%
(Genre combinations: 951 distinct; 95% of movies have an exact genre twin)


On raw genre vectors they **never** disagree on the top result in this dataset. Lower down the lists differ slightly (~95% overlap). The reason: for 0/1 vectors, squared Euclidean distance is just the *number of mismatched genres*. Most movies have an exact genre twin (distance 0, cosine 1), so both metrics pick it. For the rest, the movie with the fewest mismatches is also among cosine's best matches.

**(c) On unnormalized tag TF-IDF vectors (`norm=None`).** Here vector *length* varies a lot: a movie with 100+ tags has a much longer vector than one with 2 tags. This is where the two metrics split:

In [47]:
tag_tfidf_raw = TfidfVectorizer(stop_words="english", min_df=2, norm=None)      # no length normalization
T_raw = tag_tfidf_raw.fit_transform(movies["tags_text"])
lengths = np.sqrt(np.asarray(T_raw.multiply(T_raw).sum(axis=1)).ravel())
tagged = np.where(lengths > 0)[0]
print(f"Vector length of tagged movies: min {lengths[tagged].min():.1f} | median {np.median(lengths[tagged]):.1f} "
      f"| max {lengths[tagged].max():.1f}")

dis_t, ov_t, euc_pick = compare_top_results(T_raw, tagged)
print(f"Seeds where the top result differs: {dis_t.sum():,} of {len(tagged):,} ({dis_t.mean():.1%})")
print(f"Average overlap of the two top-10 lists: {ov_t.mean():.1%}")
print(f"Euclidean's top pick has an EMPTY tag vector: {(lengths[euc_pick] == 0).mean():.1%} of seeds "
      f"(cosine: never, since an empty vector has cosine 0 with everything)")

Vector length of tagged movies: min 6.3 | median 10.0 | max 172.4


Seeds where the top result differs: 187 of 1,306 (14.3%)
Average overlap of the two top-10 lists: 57.8%
Euclidean's top pick has an EMPTY tag vector: 8.3% of seeds (cosine: never, since an empty vector has cosine 0 with everything)


In [48]:
# A concrete case: A Bug's Life (its only tag is "pixar")
ex_seed = title_to_idx["Bug's Life, A (1998)"]
cos = cosine_similarity(T_raw[ex_seed], T_raw).ravel(); cos[ex_seed] = -np.inf
euc = euclidean_distances(T_raw[ex_seed], T_raw).ravel(); euc[ex_seed] = np.inf
print(f"Seed: {movies.loc[ex_seed, 'title']} | tags: '{movies.loc[ex_seed, 'tags_text']}' "
      f"| own vector length {lengths[ex_seed]:.2f}\n")
top_c = np.argsort(-cos, kind="stable")[:3]
top_e = np.argsort(euc, kind="stable")[:3]
print("Top 3 by COSINE:")
display(movies.iloc[top_c][["title"]].assign(cosine=cos[top_c].round(3), tag_vector_length=lengths[top_c].round(2),
                                             tags=movies.tags_text.iloc[top_c].str[:40].values))
print("Top 3 by EUCLIDEAN distance:")
display(movies.iloc[top_e][["title"]].assign(distance=euc[top_e].round(2), tag_vector_length=lengths[top_e].round(2),
                                             tags=movies.tags_text.iloc[top_e].str[:40].values))
print(f"Movies with an empty tag vector, all exactly {lengths[ex_seed]:.2f} away from the seed: {int((lengths == 0).sum()):,}")

Seed: Bug's Life, A (1998) | tags: 'pixar' | own vector length 8.57

Top 3 by COSINE:


,title,cosine,tag_vector_length,tags
0,Toy Story (1995),0.901,19.03,pixar pixar fun
2355,Toy Story 2 (1999),0.640,26.78,animation disney funny original pixar se
7039,Up (2009),0.301,28.44,adventure bechdel test:fail cartoon chil


Top 3 by EUCLIDEAN distance:


,title,distance,tag_vector_length,tags
2,Grumpier Old Men (1995),8.57,0.0,moldy old
3,Waiting to Exhale (1995),8.57,0.0,
5,Heat (1995),8.57,0.0,


Movies with an empty tag vector, all exactly 8.57 away from the seed: 8,436


**Finding:**
- **Normalized TF-IDF (what the recommender uses):** cosine and Euclidean can **never** disagree, because `‖a − b‖² = 2 − 2·cos`.
- **Raw 0/1 genres:** they never disagree on the top result here, and differ only slightly further down.
- **Unnormalized tag vectors:** they disagree on the top result for 14% of seeds, and the top-10 lists share only 58%. For 8% of seeds, Euclidean's nearest movie has an **empty** tag vector. *A Bug's Life* (tag: "pixar") is the clearest case. Cosine picks *Toy Story* and *Toy Story 2* (also "pixar"). Euclidean picks *Grumpier Old Men*, because the distance to an empty vector is just the seed's own length (8.57), while *Toy Story*'s long tag vector (19.03) puts it farther away even though it shares the tag. All 8,436 empty-vector movies tie at that distance. So Euclidean rewards *short* vectors, not *similar* ones. Cosine only looks at direction, so an empty vector scores 0 and related movies win.

That is the practical reason cosine is the standard for TF-IDF and text: it measures *what the item is about*, not *how much text it has*.

## Exercise 3 — "More like this" for a single movie (no user)
The purest item-item content-based recommendation: one seed movie in, similar movies out. No user profile at all.

In [49]:
for t in ["Toy Story (1995)", "Dark Knight, The (2008)", "Notebook, The (2004)"]:
    print(f"More like: {t} | {recommender.movies.loc[recommender.title_to_idx[t], 'genres']}")
    display(recommender.more_like_this(t, top_n=10))

More like: Toy Story (1995) | Adventure|Animation|Children|Comedy|Fantasy


,title,genres,similarity
0,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,0.883
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.873
2,The Lego Movie (2014),Action|Adventure|Animation|Children|Comedy|Fantasy,0.718
3,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,0.700
4,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.700
5,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,0.700
6,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,0.700
7,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,0.700
8,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,0.700
9,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,0.700


More like: Dark Knight, The (2008) | Action|Crime|Drama|IMAX


,title,genres,similarity
0,Need for Speed (2014),Action|Crime|Drama|IMAX,0.700
1,Batman Begins (2005),Action|Crime|IMAX,0.675
2,"Fast Five (Fast and the Furious 5, The) (2011)",Action|Crime|Drama|Thriller|IMAX,0.653
3,"Dark Knight Rises, The (2012)",Action|Adventure|Crime|IMAX,0.644
4,Eagle Eye (2008),Action|Crime|Thriller|IMAX,0.626
5,"Good Day to Die Hard, A (2013)",Action|Crime|Thriller|IMAX,0.626
6,"Fast & Furious 6 (Fast and the Furious 6, The) (2013)",Action|Crime|Thriller|IMAX,0.626
7,"Grandmaster, The (Yi dai zong shi) (2013)",Action|Drama|IMAX,0.623
8,RoboCop (2014),Action|Crime|Sci-Fi|IMAX,0.603
9,Man of Tai Chi (2013),Action|IMAX,0.595


More like: Notebook, The (2004) | Drama|Romance


,title,genres,similarity
0,Titanic (1997),Drama|Romance,0.865
1,Harold and Maude (1971),Comedy|Drama|Romance,0.769
2,"Corrina, Corrina (1994)",Comedy|Drama|Romance,0.711
3,Sense and Sensibility (1995),Drama|Romance,0.700
4,Leaving Las Vegas (1995),Drama|Romance,0.700
5,Persuasion (1995),Drama|Romance,0.700
6,How to Make an American Quilt (1995),Drama|Romance,0.700
7,When Night Is Falling (1995),Drama|Romance,0.700
8,Bed of Roses (1996),Drama|Romance,0.700
9,Once Upon a Time... When We Were Colored (1995),Drama|Romance,0.700


**Finding:** this is the "Because you watched X" row on a streaming home page. It needs nothing but the movie's features, so it works even for a brand-new movie or a logged-out visitor. The results also show exactly how good the features are:
- ***Toy Story*** → *A Bug's Life*, *Toy Story 2* (shared "pixar" tag), then animated family films. Good.
- ***The Notebook*** → *Titanic* and *Harold and Maude* (both share the word "romance" in their tags), then romantic dramas. Good.
- ***The Dark Knight*** → **_Need for Speed_ comes first**, a car-racing film, simply because its genre string (Action|Crime|Drama|IMAX) is identical. *Batman Begins* comes 2nd on genres alone, because it has no tags at all in this small dataset. *The Dark Knight Rises* does share the "superhero" tag, but its genre string is different (Adventure instead of Drama), and genres carry 70% of the score, so it only reaches 4th. This is the "capped by feature quality" weakness from Phase 7, seen directly: with richer features (plot, cast, director) the Batman films would win easily.

## Exercise 4 — New-user cold start

In [50]:
new_user_id = int(ratings.userId.max()) + 1
print(f"Brand-new user {new_user_id}: {(ratings.userId == new_user_id).sum()} ratings "
      f"-> profile = {recommender.build_user_profile(new_user_id)}")

# The same problem hides inside the data: users who rated movies but never gave >= 4 stars
no_likes = sorted(set(ratings.userId) - set(ratings.loc[ratings.rating >= LIKE_THRESHOLD, "userId"]))
for uid in no_likes:
    r = ratings.loc[ratings.userId == uid, "rating"]
    print(f"Existing user {uid}: {len(r)} ratings, highest {r.max()} -> nothing >= {LIKE_THRESHOLD} "
          f"-> profile = {recommender.build_user_profile(uid)}")

print(f"\nWhat recommend_for_user() falls back to for user {new_user_id}:")
display(recommend_for_user(new_user_id))

Brand-new user 611: 0 ratings -> profile = None
Existing user 442: 20 ratings, highest 2.5 -> nothing >= 4.0 -> profile = None

What recommend_for_user() falls back to for user 611:


,title,genres,score,because_you_liked
0,"Shawshank Redemption, The (1994)",Crime|Drama,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
1,Forrest Gump (1994),Comedy|Drama|Romance|War,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
2,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
3,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
4,"Matrix, The (1999)",Action|Sci-Fi|Thriller,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
5,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
6,Fight Club (1999),Action|Crime|Drama|Thriller,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
7,Schindler's List (1993),Drama|War,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
8,Star Wars: Episode V - The Empire Strikes Back (1980),Action|Adventure|Sci-Fi,NaN,cold start: no liked movies -> most-liked movies overall (not cont...
9,Braveheart (1995),Action|Drama|War,NaN,cold start: no liked movies -> most-liked movies overall (not cont...


**Why content-based filtering alone can't solve this:** a user profile is the average of the vectors of movies the user liked. With zero ratings there is nothing to average. The profile is undefined (`None`), so there is no direction in feature space to compare movies against. Item features are still available, but without *any* signal about the user, every movie is equally plausible. The same happens to an existing user who never rated anything ≥ 4: with the "liked = rating ≥ 4" rule, they are cold too. (Using ratings relative to the user's own average would rescue them, but a brand-new user still has nothing.)

**What to recommend instead:**
1. **Popularity fallback** (implemented above): the most-liked movies overall. Not personalized, but a safe start. Note that this uses *other users'* data, so it's no longer content-based.
2. **Onboarding questions:** ask the new user to pick a few favourite genres or movies at sign-up. That gives a tiny explicit profile, and content-based filtering works again immediately (demo below).
3. **Context as the query** (Phase 1): device, time, location or the page they arrived from can steer a first guess before any history exists.
4. Switch to personalized recommendations as soon as the first few ratings or clicks arrive.

In [51]:
def recommend_for_new_user(favourite_genres, model=recommender, top_n=10, min_likes=0):
    """Onboarding: build a profile from the genres a new user picked, then score like any other user.
    min_likes > 0 keeps only movies liked by at least that many users (well-known titles)."""
    g_profile = model.genre_vec.transform(["|".join(favourite_genres)]).toarray()
    if model.year_weight > 0:
        g_profile = np.hstack([g_profile, np.zeros((1, 1), dtype=np.float32)])
    t_profile = np.zeros((1, model.T.shape[1]), dtype=np.float32)          # no tags known yet
    scores = model._similarity(g_profile, t_profile).ravel().astype(float)
    if min_likes > 0:
        likes = model.ratings[model.ratings.rating >= model.like_threshold].groupby("movieId").size()
        keep = model.movieid_to_idx[likes[likes >= min_likes].index].to_numpy()
        mask = np.full(len(scores), -np.inf)
        mask[keep] = 0
        scores += mask
    top = np.argsort(-scores, kind="stable")[:top_n]
    out = model.movies.iloc[top][["title", "genres"]].copy()
    out["score"] = scores[top].round(3)
    return out.reset_index(drop=True)

print("New user picked: Sci-Fi + Thriller")
pd.concat({"pure content-based": recommend_for_new_user(["Sci-Fi", "Thriller"])[["title", "genres"]],
           "only movies liked by >= 20 users": recommend_for_new_user(["Sci-Fi", "Thriller"], min_likes=20)[["title", "genres"]]},
          axis=1)

New user picked: Sci-Fi + Thriller


pure content-based                  only movies liked by >= 20 users                        
                                    title           genres                            title                  genres
0        Island of Dr. Moreau, The (1996)  Sci-Fi|Thriller                   Gattaca (1997)   Drama|Sci-Fi|Thriller
1                           Sphere (1998)  Sci-Fi|Thriller                        Pi (1998)   Drama|Sci-Fi|Thriller
2                      Retroactive (1997)  Sci-Fi|Thriller      The Butterfly Effect (2004)   Drama|Sci-Fi|Thriller
3                          Runaway (1984)  Sci-Fi|Thriller                Ex Machina (2015)   Drama|Sci-Fi|Thriller
4    Incredible Shrinking Man, The (1957)  Sci-Fi|Thriller              Blade Runner (1982)  Action|Sci-Fi|Thriller
5                       Brainstorm (1983)  Sci-Fi|Thriller           Terminator, The (1984)  Action|Sci-Fi|Thriller
6                  Time After Time (1979)  Sci-Fi|Thriller               Matrix, The (1999)  Action|Sci-Fi|Thriller
7                   Final Cut, The (2004)  Sci-Fi|Thriller                  Predator (1987)  Action|Sci-Fi|Thriller
8                      Futureworld (1976)  Sci-Fi|Thriller               Equilibrium (2002)  Action|Sci-Fi|Thriller
9  Timecrimes (Cronocrímenes, Los) (2007)  Sci-Fi|Thriller     X-Men: The Last Stand (2006)  Action|Sci-Fi|Thriller

Pure content-based onboarding works, but it surfaces the movies whose genre string is *exactly* "Sci-Fi|Thriller", which are mostly obscure titles. Restricting to well-known movies (a popularity filter, again borrowed from other users' data) gives a much better first impression. That combination is a small preview of why real systems end up **hybrid**.

## Exercise 5 — TF-IDF ablation: drop IDF, use TF only
`use_idf=False` keeps everything else identical (same tokens, same L2 normalization, same scoring), so only the IDF factor changes. A clean ablation changes one thing at a time.

In [52]:
tf_only = ContentBasedRecommender(DATA_DIR, use_idf=False, precompute_similarity=False).fit()
display(overlap_at_k(base, tf_only).to_frame("TF-IDF vs TF-only").T)

# Are the recommended movies more "generic" without IDF?  (catalog share: Drama ~0.45, Film-Noir ~0.01)
genre_freq = genres_dummies[GENRE_COLS].mean()
summary = {}
for name, model in [("TF-IDF", base), ("TF only", tf_only)]:
    recs = pd.concat([model.recommend_for_user(u, explain=False) for u in all_users])
    summary[name] = {
        "recs containing Drama": recs.genres.str.contains("Drama", regex=False).mean(),
        "recs containing Comedy": recs.genres.str.contains("Comedy", regex=False).mean(),
        "avg catalog share of a recommended movie's genres": recs.genres.str.split("|").explode().map(genre_freq).mean(),
    }
print(f"All {len(all_users)} users' top-10 lists:")
display(pd.DataFrame(summary).T.round(3))

cmp = pd.concat({"TF-IDF": base.more_like_this("Chinatown (1974)")[["title", "genres"]],
                 "TF only": tf_only.more_like_this("Chinatown (1974)")[["title", "genres"]]}, axis=1)
print("More like 'Chinatown (1974)' (Crime|Film-Noir|Mystery|Thriller):")
display(cmp)

print(f"User {horror_user} (horror fan):")
display(pd.concat({"TF-IDF": base.recommend_for_user(horror_user, explain=False)[["title", "genres"]],
                   "TF only": tf_only.recommend_for_user(horror_user, explain=False)[["title", "genres"]]}, axis=1))

,mean overlap@10,users with identical top-10,users with <=5 shared
TF-IDF vs TF-only,0.769,0.139,0.115


All 610 users' top-10 lists:


,recs containing Drama,recs containing Comedy,avg catalog share of a recommended movie's genres
TF-IDF,0.775,0.554,0.244
TF only,0.839,0.556,0.252


More like 'Chinatown (1974)' (Crime|Film-Noir|Mystery|Thriller):


TF-IDF                                                                                  TF only  \
                                            title                                   genres                                           title   
0                    Devil in a Blue Dress (1995)         Crime|Film-Noir|Mystery|Thriller                    Devil in a Blue Dress (1995)   
1                        L.A. Confidential (1997)         Crime|Film-Noir|Mystery|Thriller                        L.A. Confidential (1997)   
2                           House of Games (1987)         Crime|Film-Noir|Mystery|Thriller                           House of Games (1987)   
3  Pure Formality, A (Pura formalità, Una) (1994)         Crime|Film-Noir|Mystery|Thriller  Pure Formality, A (Pura formalità, Una) (1994)   
4                         Mulholland Drive (2001)   Crime|Drama|Film-Noir|Mystery|Thriller                         Mulholland Drive (2001)   
5                       Too Late for Tears (1949)   Crime|Drama|Film-Noir|Mystery|Thriller                                 Sin City (2005)   
6                                 Sin City (2005)  Action|Crime|Film-Noir|Mystery|Thriller                       Too Late for Tears (1949)   
7                                    Laura (1944)                  Crime|Film-Noir|Mystery                      Usual Suspects, The (1995)   
8                           Big Sleep, The (1946)                  Crime|Film-Noir|Mystery                                    Laura (1944)   
9                           Third Man, The (1949)               Film-Noir|Mystery|Thriller                        Dial M for Murder (1954)   

                                            
                                    genres  
0         Crime|Film-Noir|Mystery|Thriller  
1         Crime|Film-Noir|Mystery|Thriller  
2         Crime|Film-Noir|Mystery|Thriller  
3         Crime|Film-Noir|Mystery|Thriller  
4   Crime|Drama|Film-Noir|Mystery|Thriller  
5  Action|Crime|Film-Noir|Mystery|Thriller  
6   Crime|Drama|Film-Noir|Mystery|Thriller  
7                   Crime|Mystery|Thriller  
8                  Crime|Film-Noir|Mystery  
9                   Crime|Mystery|Thriller

User 571 (horror fan):


TF-IDF                                                         TF only                                
                      title                          genres                           title                          genres
0               Cujo (1983)                 Horror|Thriller                     Cujo (1983)                 Horror|Thriller
1               Fido (2006)          Comedy|Horror|Thriller                 Scream 3 (2000)  Comedy|Horror|Mystery|Thriller
2     My Demon Lover (1987)                   Comedy|Horror                     Fido (2006)          Comedy|Horror|Thriller
3  Shaun of the Dead (2004)                   Comedy|Horror                 Rose Red (2002)         Horror|Mystery|Thriller
4          Session 9 (2001)                 Horror|Thriller                Session 9 (2001)                 Horror|Thriller
5           Scream 3 (2000)  Comedy|Horror|Mystery|Thriller           My Demon Lover (1987)                   Comedy|Horror
6      Babadook, The (2014)           Drama|Horror|Thriller           Evil Dead, The (1981)         Fantasy|Horror|Thriller
7           Rose Red (2002)         Horror|Mystery|Thriller            Babadook, The (2014)           Drama|Horror|Thriller
8                Bug (2007)           Drama|Horror|Thriller            Relative Fear (1994)                 Horror|Thriller
9      Relative Fear (1994)                 Horror|Thriller  In the Mouth of Madness (1995)                 Horror|Thriller

**Finding:** dropping IDF changes about a quarter of every user's list (77% overlap on average; only 14% of users keep an identical top-10), even though the items' features didn't change at all. Without IDF every genre counts the same, so the most common genre pulls as hard as the rarest:
- **Recommendations drift towards Drama:** 84% of recommended movies contain Drama without IDF vs. 78% with IDF.
- **Distinctive genres lose their pull:** for *Chinatown* (Crime|Film-Noir|Mystery|Thriller), TF-only lets *The Usual Suspects* and *Dial M for Murder* (no Film-Noir) into the top 10. With IDF, the rare **Film-Noir** label keeps classic noirs like *The Big Sleep* and *The Third Man* in the list.
- For the horror fan, the top items stay similar but the order and the tail change.

IDF is doing what Phase 4 said: sharing a *rare* feature (Film-Noir, or a tag like "cyberpunk") counts for much more than sharing "Drama", which matches intuition. Two noirs are far more alike than two dramas.

In [53]:
del base, with_year, tf_only
gc.collect();

---
# ✔ Deliverables Checklist

- [x] **Phase 1:** real-world query/item table (5 platforms) + zero-query-problem explanation
- [x] **Phase 2:** feedback matrix (610 × 9,724), sparsity calculation, implicit binary version, explicit-vs-implicit write-up
- [x] **Phase 3:** full item-profile table (multi-hot genres + scaled year) + single-item feature vector explanation
- [x] **Phase 4:** TF-IDF matrix + top-weighted terms interpretation (genres and tags)
- [x] **Phase 5:** top-10 similar-item results for 2 seed movies (genres only and genres + tags), with sanity checks
- [x] **Phase 6:** two distinct user profiles with differing top-10 recommendations
- [x] **Phase 7:** strengths/weaknesses table + filter-bubble write-up (with a measured demo)
- [x] **Phase 8:** working end-to-end `recommend_for_user()` with explanations, tested on 3 users, timed, scaling note
- [x] **Exercises:** all 5 attempted (year feature, cosine vs Euclidean, more-like-this, new-user cold start, TF-IDF ablation)
- [x] **Code:** notebook runs top to bottom without errors

### Fixes made to the readme's reference code
| Where | Issue | Fix |
|---|---|---|
| Phase 4 | Default tokenizer splits `Sci-Fi`, `Film-Noir`, and turns `(no genres listed)` into fake terms | Tokenizer that splits on `\|` and drops the placeholder |
| Phase 5 | `[1:top_n+1]` assumes the seed ranks first; with tied scores the seed can appear in its own list | Exclude the seed by index |
| Phase 5 | Full similarity matrix in float64 takes ~760 MB | float32 (~380 MB), and freed when no longer needed |
| Phase 6 | `weights = rating.values[:len(idxs)]` misaligns weights if any movie is skipped | Filter the DataFrame first so IDs and weights stay paired |
| Phase 4.6 (data) | Most common tag "in netflix queue" is a to-watch list, not movie content | Removed before building tag vectors |